# tools

> The hands: what the application under the agent must provide, and every tool built on top of it.

`Host` is the whole dependency this package has on the world. Everything else here is derived from it. The tools the model is given, the skills it can read, the extensions a user can add, and the sub-agents it can delegate to. A host that satisfies the signatures but not the contracts is a host that quietly hands an agent the whole filesystem. The docstrings in `Host` are the specification.

In [ ]:
#| default_exp tools

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import tempfile
from fastcore.test import test_eq, test_fail, expect_fail
from ramabana.testing import MemHost, FakeBackend

In [ ]:
#| export
import ast, concurrent.futures, functools, json, mimetypes, os, re, runpy, shutil, threading, time, uuid
from base64 import b64decode
from dataclasses import dataclass, field
from pathlib import Path
from fastcore.basics import AttrDict, ifnone
from fastcore.xtras import detect_mime
from fastcore.docments import frontmatter
from fastcore.foundation import L
from fastcore.parallel import parallel, startthread
from ramabana.core import AgentError, agent_err, spec_caps
from ramabana.runtime import Run, current_run, run_context

## The host

`Host` is the application under an agent, as an interface. Every method may raise, and the tools catch rather than let an exception end a turn. A capability that cannot be supported raises `NotImplementedError`, which `tools_for` reads as "do not offer this tool". An agent told about a tool that always fails is worse off than one never told about it.

In [ ]:
#| export
MAX_GREP_HITS = 60  # exact matches one `grep` returns. Part of the Host contract
MAX_API = 200       # public names one `public_api` listing returns. Part of the same contract

class Hit(AttrDict):
    "One `Host.search` hit: path, line, symbol, text."
    def __init__(self, path, line=1, symbol='', text=''):
        super().__init__(path=path, line=line, symbol=symbol, text=text)
    def __repr__(self): return f'{self.path}:{self.line}  {self.symbol}  {self.text}'

In [ ]:
#| export
class Host:
    "The application under an agent. Every method may raise. Absent ones raise `NotImplementedError`."


    @property
    def roots(self):
        "The open folders, as absolute paths. The agent is told about these and confined to them."
        raise NotImplementedError

    @property
    def added_roots(self):
        "The roots opened after this host was built, which a resumed session must not inherit."
        return []

    def add_root(self, path):
        "Open another folder, and return it resolved. Widening the write boundary, so hosts may refuse."
        raise NotImplementedError

    def check(self, path, must_exist=False, reading=False):
        """The single chokepoint: resolve `path`, refuse anything outside `roots`, return a `Path`.

        `reading=True` says the caller will only *read* what comes back, and it is the one case a
        host may answer for a path outside `roots`. See `LocalHost(read_outside=)`.
        """
        raise NotImplementedError

    def walk(self):
        "Every readable file under the open folders."
        raise NotImplementedError

    def read(self, path):
        "One file's text, or None when it cannot be read."
        raise NotImplementedError

    def write(self, path, text):
        "Write `text` to `path`, through the same sandbox `check` enforces. Returns the path written."
        raise NotImplementedError

    def text_at(self, path):
        "One file as a single diffable document, `''` when it does not exist yet, None on error."
        raise NotImplementedError


    def search(self, query, limit=20):
        "Search the code index for `query`, returning `Hit`s. Semantic if an index exists, literal if not."
        raise NotImplementedError

    def peers(self, path, line, limit=20):
        "Code shaped like whatever is defined at `path`:`line`. Every place a pattern was already used."
        raise NotImplementedError

    def symbols(self, path):
        "The defs and classes in one file, as `Hit`s whose `score` is the indent depth."
        raise NotImplementedError

    def public_api(self, package, limit=MAX_API):
        """Every public name `package` exports, as `Hit`s whose `symbol` is the qualified name.

        Raise rather than return `[]` when there is no index: "this package exports nothing"
        and "nothing could be looked up" must not reach the model as the same answer.
        An empty `package` is the harmless probe `code_tools` uses, and answers `[]`.
        """
        raise NotImplementedError

    def grep(self, pattern, path_filter='', regex=True, ignore_case=False, limit=MAX_GREP_HITS):
        "Every line under the open folders matching `pattern` exactly, as `Hit`s. None means 'I have no exact matcher'."
        return None

    @property
    def search_note(self):
        "Which engine answered, and anything it wants to say about why. Shown when a search finds nothing."
        return ''


    def web_search(self, query, n=20):
        "Search the web. Returns objects with `.title` and `.url`."
        raise NotImplementedError

    def read_url(self, url, remember=True):
        "One page as markdown. `remember=False` keeps sensitive/low-quality results ephemeral."
        raise NotImplementedError

    def research(self, query):
        "Search and read the top results into one cited digest. Slower than `web_search`."
        raise NotImplementedError

    @property
    def research_note(self): return ''


    def memory_search(self, query, limit=8):
        "Search remembered pages as whole tree sections, returning structured rows."
        raise NotImplementedError

    def memory_tree(self, document=''):
        "The heading tree for remembered documents. An empty document lists every root."
        raise NotImplementedError

    def memory_read(self, node_id):
        "Read one remembered section and its children by stable node id."
        raise NotImplementedError

    def memory_topics(self, limit=12):
        "Labelled semantic clusters across remembered research."
        raise NotImplementedError

    def memory_forget(self, doc_id):
        "Purge one remembered document and all derived tree/chunk/vector data."
        raise NotImplementedError

    def ask(self, question, ref=None, instruction='', **kw):
        "Answer `question` out of remembered research, with citations, as a dict."
        raise NotImplementedError


    # What the agent arranged to read *later*: a watch is a job the host re-runs on an
    # interval, `poll` is the tick, and a reminder is a watch that files its own text.
    def remember(self, text, title=None, tags=()):
        "File `text` into durable memory as a note. Returns the document record."
        raise NotImplementedError

    def watch(self, target, action='remind', every='1d', note=None, **params):
        "Register a recurring job. `target` is a URL, a query, or the text of a reminder."
        raise NotImplementedError

    def watches(self, due_only=False):
        "Every registered watch, soonest first. `due_only` keeps the ones that have come due."
        raise NotImplementedError

    def unwatch(self, watch_id):
        "Delete one watch. Whatever it already filed stays in memory."
        raise NotImplementedError

    def poll(self):
        "Run every watch that is due and report what fired. One failing watch must not stop the rest."
        raise NotImplementedError

    @property
    def watch_actions(self):
        "The `action` values this host's `watch` will accept."
        return ('remind',)


    # No notebook representation here: exhash addresses cells by path and id without one,
    # so only the two operations that need to know what a notebook *is* are delegated.
    def nb_cells(self, path):
        "`[(id, cell_type, first_line)]` for one notebook."
        raise NotImplementedError

    def nb_add_cell(self, path, source, index=-1, cell_type='code'):
        "Insert a cell (-1 appends), creating the notebook if needed. Returns the new cell's id."
        raise NotImplementedError


    def run_python(self, code):
        """Run `code` in the user's live namespace under whatever restrictions the host imposes.

        The contract the agent is briefed on, and the host's to enforce: read anything, bind
        results to new names, never rebind or delete the owner's.
        """
        raise NotImplementedError

    def inspect_python(self, code, scope='isolated'):
        """Run `code` against the live namespace without touching what the user has.

        Two scopes, both protecting the owner's variables, by different means:

        - `'isolated'` runs in an allowlist sandbox on a *copy*. Attribute reads and builtins
          work. Most library method calls are refused. The default, and it needs no trust.
        - `'overlay'` runs the real interpreter against the real namespace under an AST policy:
          read anything, bind names in the agent's own layer, never delete, rebind or mutate
          the owner's. `list(df.columns)` works here. In the sandbox it does not.

        A host may refuse `'overlay'`. See `scopes`.
        """
        raise NotImplementedError

    @property
    def scopes(self):
        "The scopes `inspect_python` will actually honour, most trusted last."
        return ('isolated',)

    @property
    def kernel_kind(self):
        "What runs the live namespace. `'ipymini'` inspects while a cell is busy. Anything else queues."
        return 'ipykernel'

    @property
    def concurrent(self): return self.kernel_kind == 'ipymini'

    def list_vars(self):
        "What is in the live namespace: name, type, and a short value, one per line."
        raise NotImplementedError

    def terminal_text(self, lines=200):
        "What the IDE's terminal has printed. Read-only: this shows what the user ran, it cannot run anything."
        raise NotImplementedError


    def run_cmd(self, command, cwd=None, timeout=120):
        """Run `command` in a shell and return `(exit_code, combined_output)`.

        Contract a host must keep, because the tool trusts it:

        - `cwd` is resolved through `check`. Confining the *working directory* is not confining
          the command, which is why `run_shell` is in `WRITE_TOOLS` and goes to a person.
        - stdout and stderr come back interleaved, in one string, in order.
        - `timeout` is enforced and the whole process *group* is killed on expiry.
        - A failed command returns a non-zero exit code rather than raising.
        - An **empty** command is a no-op returning `(0, '')` and must not spawn anything.
          That is how `tools_for` asks "can you run commands?" without running one.
        """
        raise NotImplementedError

    @property
    def shell_note(self):
        "How commands are run here (the interpreter, the default directory), or why they are not."
        return ''


    @property
    def capabilities(self):
        """Which tool groups this host supports, for the ones it can answer without proving it.

        `{group: bool}` for the groups this host *knows* its answer to. Anything absent is
        probed with a harmless call instead. The names are the tool group functions:
        `notebook`, `web`, `memory`, `watch`, `session`, `shell`.
        """
        return {}


    @property
    def approvals(self):
        "The `Approvals` this host uses to put a write in front of a person, or None to approve everything."
        return None

    def note(self, text):
        "Tell the user something out of band (a status line). Never blocks. A host may drop it."
        pass

A `Hit` is the one shape every search backend returns, whether the index behind it is semantic or a literal scan.

In [ ]:
Hit('nbs/02_tools.ipynb', 42, 'tools_for', 'def tools_for(host, get_skills=None, extra=()):')

```python
nbs/02_tools.ipynb:42  tools_for  def tools_for(host, get_skills=None, extra=()):
```

`NullHost` is a host with nothing behind it, and it is the reference implementation of "absent". The harness runs bare on it, which is how the probing in `tools_for` gets tested without a real application.

In [ ]:
#| export
class NullHost(Host):
    "A host with nothing behind it: every capability absent. The harness runs bare in a test."

    def __init__(self, roots=()): self._roots = [str(r) for r in roots]

    @property
    def roots(self): return self._roots
    @property
    def added_roots(self): return list(getattr(self, '_added', []))
    def add_root(self, path):
        p = str(Path(path).expanduser().resolve())
        if p not in self._roots:
            self._roots.append(p); self.__dict__.setdefault('_added', []).append(p)
        return p

    def check(self, path, must_exist=False, reading=False):
        from pathlib import Path
        return Path(path)

    def walk(self): return []
    def read(self, path): return None
    def write(self, path, text): raise NotImplementedError
    def text_at(self, path): return None
    def search(self, query, limit=20): return []
    def peers(self, path, line, limit=20): return []
    def symbols(self, path): return []
    def web_search(self, query, n=20): return []
    def read_url(self, url, remember=True): return None
    def research(self, query): return ''
    def memory_search(self, query, limit=8): raise NotImplementedError
    def memory_tree(self, document=''): raise NotImplementedError
    def memory_read(self, node_id): raise NotImplementedError
    def memory_topics(self, limit=12): raise NotImplementedError
    def memory_forget(self, doc_id): raise NotImplementedError
    def remember(self, text, title=None, tags=()): raise NotImplementedError
    def watch(self, target, action='remind', every='1d', note=None, **params): raise NotImplementedError
    def watches(self, due_only=False): raise NotImplementedError
    def unwatch(self, watch_id): raise NotImplementedError
    def poll(self): raise NotImplementedError
    def nb_cells(self, path): raise NotImplementedError
    def nb_add_cell(self, path, source, index=-1, cell_type='code'): raise NotImplementedError
    def run_python(self, code): raise NotImplementedError
    def run_cmd(self, command, cwd=None, timeout=120): raise NotImplementedError
    def inspect_python(self, code, scope='isolated'): raise NotImplementedError
    def list_vars(self): raise NotImplementedError
    def terminal_text(self, lines=200): raise NotImplementedError

In [ ]:
h = NullHost(['/proj'])
h.roots, h.walk(), h.read('/proj/a.py')

(['/proj'], [], None)

Two properties are answered rather than raised, because the agent is briefed on them before it calls anything: which inspection scopes are honoured, and whether the kernel can run an inspection while a cell is busy. "Read the dataframe" is good advice under one and a way to hang the session under the other.

In [ ]:
h.scopes, h.kernel_kind, h.concurrent

(('isolated',), 'ipykernel', False)

In [ ]:
with expect_fail(NotImplementedError): h.run_python('1+1')
h.approvals is None

True

## A host over real folders

`LocalHost` is the reference implementation: enough of a host to run the agent from a terminal, from an MCP server or from a test, with no IDE anywhere. It is also where the sandbox actually lives. `check` resolves `..` and symlinks *before* comparing against the open folders. Every other method may assume its argument was approved.

In [ ]:
#| export
SANDBOX = 'path is outside the open folders'
SECRET = 'path holds credentials and is never read'
NO_ROOTS = 'no folders are open, so no path is inside them'

#: Credential-shaped paths refused even with `read_outside`. `fnmatch` on the resolved path.
DENY = ('*/.ssh/*', '*/.aws/*', '*/.gnupg/*', '*/.config/gcloud/*', '*/.netrc',
        '*/.git-credentials', '*/.codex/auth.json', '*/.claude/.credentials.json',
        '*/.env', '*/.env.*', '*/id_rsa*', '*/id_ed25519*', '*.pem', '*.key', '*.p12')
SKIP_DIRS = frozenset({'.git', '.hg', '.svn', '__pycache__', '.venv', 'venv', 'node_modules',
                       '.ipynb_checkpoints', '.pytest_cache', '.mypy_cache', '_docs', '_proc',
                       'dist', 'build', '.quarto', '.idea', '.attic'})
SKIP_SUFFIXES = frozenset({'.pyc', '.pyo', '.so', '.dylib', '.dll', '.a', '.o', '.zip', '.gz',
                           '.whl', '.png', '.jpg', '.jpeg', '.gif', '.webp', '.pdf', '.parquet',
                           '.sqlite', '.db', '.bin', '.safetensors', '.gguf'})
MAX_FILE = 2_000_000      # bytes. A file larger than this is data, not source
MAX_VARS = 200
LD_CHARS = 4000           # of a page's JSON-LD to keep. Enough for a product, not a catalogue

_LD = re.compile(r'<script[^>]+application/ld\+json[^>]*>(.*?)</script>', re.S | re.I)


def denied(path, patterns=DENY):
    "Whether `path` is one of the things reading outside the open folders still must not open."
    from fnmatch import fnmatch
    s = Path(path).as_posix()
    return any(fnmatch(s, pat) for pat in patterns)

def _md_doc(d):
    "One of fossick's document readers' results as markdown: the fields it has, then its text."
    if isinstance(d, str): return d
    if not isinstance(d, dict): return str(d or '')
    head = [f'**{k}**: {v}' for k in ('title', 'authors', 'published', 'channel', 'duration', 'link')
            if (v := d.get(k)) not in (None, '', [], {})]
    body = next((str(d[k]) for k in ('source', 'text', 'content', 'summary') if d.get(k)), '')
    return '\n'.join(head + [''] + [body]).strip() if head else body.strip()

def _fuse(legs, limit):
    "Merge ranked `Hit` lists with `litesearch.rrf_all`. Identity is `path:line`."
    legs = [list(l) for l in legs if l]
    if not legs: return []
    if len(legs) == 1: return legs[0][:limit]
    from litesearch import rrf_all   # imported here: it pulls pandas, and only fusion needs it
    by_key, lists = {}, []
    for leg in legs:
        rows = []
        for h in leg:
            key = f'{h.path}:{h.line}'
            by_key.setdefault(key, h)
            rows.append({'_fid': key})
        lists.append(rows)
    try: fused = rrf_all(lists, id_key='_fid', limit=limit)
    except Exception: return legs[0][:limit]   # a bad fusion degrades the ranking, never `search`
    return [by_key[r['_fid']] for r in fused if r.get('_fid') in by_key]

def ld_json(html):
    "The `schema.org` JSON-LD blocks in `html`. Where a page states its price, author or rating."
    out = []
    for m in _LD.finditer(html or ''):
        try: out.append(json.loads(m.group(1)))
        except Exception: pass
    return out

class LocalHost(Host):
    "The reference `Host`: enough of one to run the agent from a terminal, an MCP server or a test."

    def __init__(self,
                 roots=('.',),          # the folders the agent is confined to
                 ns=None,               # the live namespace. A fresh dict when None
                 approvals=None,        # an `Approvals`, or None to gate nothing
                 note=None,             # callable for out-of-band status lines
                 web=True,              # wire the web tools to fossick when it is installed
                 index=True,            # start a Kosha sync for every open root
                 graph=False,           # build Kosha's call graph during sync
                 rerank=True,           # reorder Kosha's hits with its flashrank cross-encoder
                 rerank_model=None,     # flashrank model name. None is its fast default
                 read_outside=False,    # let read-only tools name any path on this machine
                 deny=DENY):            # what `read_outside` still refuses to open
        self._roots = [str(Path(r).expanduser().resolve()) for r in roots]
        self._added_roots = []         # opened later, and not inherited by a resumed session
        self.ns = ifnone(ns, {'__name__': '__main__'})
        self._approvals, self._note, self.web = approvals, note, web
        self.read_outside, self.deny = bool(read_outside), tuple(deny or ())
        self.transcript = []           # what this process has printed, for `read_terminal`
        self._indexes, self._index_errors, self._index_thread = [], [], None
        self._pending = list(self._roots)     # roots whose sync has not returned yet
        self.rerank, self.rerank_model, self._rerank_note = bool(rerank), rerank_model, ''
        self.graph = graph
        if index: self.sync_index()

    def sync_index(self, wait=False, force=False):
        "Run `Kosha.sync` for every open root, once, in a daemon thread. Each root publishes as it returns."
        if self._index_thread is None or not self._index_thread.is_alive():
            def run():
                try:
                    os.environ.setdefault('TQDM_DISABLE', '1')  # kosha's tqdm even with verbose=False
                    from kosha import Kosha
                except Exception as e:
                    self._index_errors.append(agent_err(e)); self._pending = []; return
                for root in list(self._roots):
                    try:
                        k = Kosha(dir=Path(root), busy_timeout=30000)
                        k.sync(dir=Path(root), verbose=False, force=force, pyproject=True, graph=self.graph)
                        self._indexes.append(k)
                    except Exception as e: self._index_errors.append(agent_err(e))
                    finally:
                        try: self._pending.remove(root)
                        except ValueError: pass
            self._index_thread = startthread(run, daemon=True)
            self._index_thread.name = 'ramabana-kosha-sync'
        if wait: self._index_thread.join()
        return self

    @property
    def index_ready(self):
        "Whether *every* open folder is indexed. `indexed` is the per-folder answer `search` uses."
        return bool(self._indexes) and not self._pending

    @property
    def indexed(self):
        "The folders whose index is built and searchable now. The rest are still syncing."
        return [str(getattr(k, 'root', '')) for k in list(self._indexes)]

    def wait_index(self, timeout=None):
        "Wait for the automatic Kosha sync. Returns whether semantic search is ready."
        if self._index_thread is not None: self._index_thread.join(timeout)
        return self.index_ready

    @property
    def roots(self): return list(self._roots)

    @property
    def added_roots(self):
        "Roots opened after construction. `/resume` lapses these, and says that it did."
        return list(self._added_roots)

    def add_root(self, path):
        """Open another folder for reading and writing, and index it. Returns it resolved.

        The one operation that widens the write boundary of a running session, so it refuses
        anything that is not already a directory rather than creating one.
        """
        p = Path(path).expanduser().resolve()
        if not p.exists(): raise AgentError(f'no such folder: {p}')
        if not p.is_dir(): raise AgentError(f'not a folder, so it cannot be a root: {p}')
        if str(p) in self._roots: return str(p)
        self._roots.append(str(p)); self._added_roots.append(str(p))
        self._pending.append(str(p))
        try: self.sync_index()
        except Exception as e: self._index_errors.append(agent_err(e))
        return str(p)

    def check(self, path, must_exist=False, reading=False):
        "Resolve `path`. Refuse outside `roots` (unless `read_outside` and `reading`). Walks stay confined."
        p = Path(path).expanduser()
        if not self._roots: raise AgentError(f'{NO_ROOTS}: {p}')  # empty roots must refuse, not IndexError
        if not p.is_absolute(): p = Path(self._roots[0])/p
        p = p.resolve()  # collapse `..` and out-of-root symlinks before comparing
        if not any(p == Path(r) or Path(r) in p.parents for r in self._roots):
            if not (reading and self.read_outside): raise AgentError(f'{SANDBOX}: {p}')
            if denied(p, self.deny): raise AgentError(f'{SECRET}: {p}')
        if must_exist and not p.exists(): raise AgentError(f'no such file: {p}')
        return p

    @property
    def roots_note(self):
        "How paths are resolved here, in one line, for the briefing and a status bar."
        n = len(self._roots)
        return (f'{n} folder(s); reads may name any path on this machine, writes may not'
                if self.read_outside else f'{n} folder(s); nothing outside them is readable')

    def _walk(self, root):
        "Files under `root`, skipping the same generated dirs/suffixes `grep` covers."
        try:
            from rgapi import fd
            rows = fd(root=root, skip_dir=sorted(SKIP_DIRS), max_filesize=MAX_FILE,
                      exclude=[f'*{s}' for s in sorted(SKIP_SUFFIXES)])
            for p in rows:
                p = Path(p)
                if not p.is_absolute(): p = Path(root)/p
                if p.is_symlink() or not p.is_file(): continue
                yield p
            return
        except Exception: pass
        for p in sorted(Path(root).rglob('*')):
            if any(part in SKIP_DIRS for part in p.parts): continue
            if not p.is_file() or p.is_symlink(): continue
            if p.suffix.lower() in SKIP_SUFFIXES: continue
            try:
                if p.stat().st_size > MAX_FILE: continue
            except OSError: continue
            yield p

    def walk(self):
        return [p for r in self._roots for p in self._walk(r)]

    def read(self, path):
        try: return self.check(path, must_exist=True, reading=True).read_text(encoding='utf-8')
        except Exception: return None

    def write(self, path, text):
        p = self.check(path)
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(str(text), encoding='utf-8')
        return str(p)

    def text_at(self, path):
        "One file as a diffable document: a notebook as its cell sources, anything else as text."
        try: p = self.check(path)
        except Exception: return None
        if not p.exists(): return ''
        if p.suffix == '.ipynb':
            try:
                from fastcore.nbio import read_nb
                return '\n\n'.join(''.join(c.source) for c in read_nb(p).cells)
            except Exception: return None
        try: return p.read_text(encoding='utf-8')
        except Exception: return None


    def _rg(self, query, limit, regex=False, ignore_case=False, path_filter='', per_file=5, every_file=False):
        "Search through `rgapi.rg`. `every_file=True` matches what `walk` yields, hidden files included."
        try: from rgapi import rg
        except Exception: return None
        pattern = query if regex else re.escape(query)
        kw = dict(case_sensitive=(False if ignore_case else None), smart_case=not ignore_case,
                  max_filesize=MAX_FILE, timeout_ms=20_000)
        if path_filter: kw['glob'] = f'*{path_filter}*'
        if every_file:
            kw.update(hidden=True, ignore=False, skip_dir=sorted(SKIP_DIRS),
                      exclude=[f'*{s}' for s in sorted(SKIP_SUFFIXES)])
        hits, counts = [], {}
        try:
            for root in self._roots:
                # pull enough rows to honour per-file caps, then trim to `limit`
                pull = None if not per_file else max(limit * 8, limit)
                for m in rg(pattern, root=root, max_results=pull, **kw):
                    if getattr(m, 'kind', 'match') != 'match': continue
                    path = Path(m.path)
                    if not path.is_absolute(): path = Path(root)/path
                    path_s = str(path)
                    if per_file:
                        n = counts.get(path_s, 0)
                        if n >= per_file: continue
                        counts[path_s] = n + 1
                    hits.append(Hit(path_s, int(m.line_number), '', (m.line or '').strip()[:200]))
                    if len(hits) >= limit: return hits
        except Exception: return None
        return hits

    def grep(self, pattern, path_filter='', regex=True, ignore_case=False, limit=MAX_GREP_HITS):
        "Exact matching through ripgrep. None when `rgapi` is unavailable. The tool then reads files itself."
        return self._rg(pattern, limit, regex=regex, ignore_case=ignore_case,
                        path_filter=path_filter, per_file=None, every_file=True)

    def _ranked(self, call, **kw):
        "One Kosha context call, reordered by its cross-encoder when reranking is on and working."
        if self.rerank:
            try: return call(rerank=True, rerank_model=self.rerank_model, **kw)
            except Exception as e:   # flashrank fetches its model on first use. Fall back once
                self.rerank = False
                self._rerank_note = f'; reranking off ({agent_err(e)})'
        return call(**kw)

    def _semantic(self, query, limit):
        "Kosha hybrid results (repo + env + graph) as the Host's stable `Hit` shape."
        indexes = list(self._indexes)
        if not indexes: return []
        out, seen = [], set()
        for k in indexes:
            try:
                rows = self._ranked(k.context, q=query, limit=limit, repo=True, env=True,
                                    graph=self.graph, columns='content,metadata')
            except Exception as e:
                self._index_errors.append(agent_err(e)); continue
            for row in rows:
                row = dict(row)
                meta = row.get('metadata') or {}
                if isinstance(meta, str):
                    try: meta = ast.literal_eval(meta)
                    except Exception: meta = {}
                path = str(meta.get('path') or row.get('path') or '')
                line = int(meta.get('lineno') or 1)
                key = (path, line)
                if key in seen: continue
                seen.add(key)
                symbol = meta.get('mod_name') or meta.get('name') or ''
                text = ' '.join(str(row.get('content') or '').split())[:240]
                out.append(Hit(path, line, str(symbol), text))
                if len(out) >= limit: return out
        return out

    def _scan(self, query, limit):
        "Every matching line, by reading the files. What is left when there is no index and no ripgrep."
        hits = []
        for p in self.walk():
            try: text = p.read_text(encoding='utf-8')
            except Exception: continue
            if query not in text: continue
            for i, line in enumerate(text.splitlines(), 1):
                if query in line:
                    hits.append(Hit(str(p), i, '', line.strip()[:200]))
                    if len(hits) >= limit: return hits
        return hits

    def search(self, query, limit=20):
        "The code index and the literal scan, fused by rank rather than tried in order."
        if not (query or '').strip(): return []
        rg = self._rg(query, limit)
        if (hits := _fuse([self._semantic(query, limit), rg or []], limit)): return hits
        return [] if rg is not None else self._scan(query, limit)

    @property
    def search_note(self):
        n, tot = len(self._indexes), len(self._roots)
        if n:
            where = f'{n} of {tot} folder(s)' if self._pending else f'{tot} folder(s)'
            return f'Kosha semantic + keyword index over {where} and environment fused with ripgrep{self._rerank_note}'
        if self._index_errors: return f'Kosha unavailable ({self._index_errors[-1]}); literal fallback'
        return 'Kosha sync in progress; literal fallback via ripgrep'

    def public_api(self, package, limit=MAX_API):
        "Kosha's public surface for `package`, `@patch`-added methods included."
        if not str(package or '').strip(): return []      # the capability probe
        indexes = list(self._indexes)
        if not indexes: raise AgentError(f'no code index: {self.search_note}')
        out, seen = [], set()
        for k in indexes:
            try: rows = k.public_api(package, meta_cols='name,mod_name,docstring,path,lineno', limit=limit)
            except Exception as e: self._index_errors.append(agent_err(e)); continue
            for row in rows:
                row = dict(row)
                name = str(row.get('mod_name') or row.get('name') or '')
                if not name or name in seen: continue
                seen.add(name)
                doc = ' '.join(str(row.get('docstring') or '').split())[:200]
                out.append(Hit(str(row.get('path') or ''), int(row.get('lineno') or 1), name, doc))
                if len(out) >= limit: return out
        return out

    def _defs(self, path):
        "Every def/class in one file as `(line, qualified_name, depth)`, by parsing rather than grepping."
        src = self.read(path)
        if src is None: return []
        try: tree = ast.parse(src)
        except SyntaxError: return []
        out = []
        def walk(node, prefix='', depth=0):
            for child in ast.iter_child_nodes(node):
                if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                    name = f'{prefix}{child.name}'
                    out.append((child.lineno, name, depth))
                    walk(child, f'{name}.', depth + 1)
        walk(tree)
        return out

    def symbols(self, path):
        "The defs and classes in one file, as `Hit`s whose `score` is the indent depth."
        p = self.check(path, reading=True)
        out = []
        for line, name, depth in self._defs(p):
            h = Hit(str(p), line, name, '')
            h.score = depth
            out.append(h)
        return out

    def peers(self, path, line, limit=20):
        "Every other place the symbol defined at `path`:`line` is mentioned."
        p = self.check(path, reading=True)
        defs = self._defs(p)
        name = next((n for ln, n, _ in sorted(defs, key=lambda d: -d[0]) if ln <= int(line)), None)
        if name is None: return []
        leaf = name.split('.')[-1]
        return [h for h in self.search(leaf, limit * 2)
                if not (str(h.path) == str(p) and h.line == int(line))][:limit]


    def nb_cells(self, path):
        from fastcore.nbio import read_nb
        nb = read_nb(self.check(path, must_exist=True, reading=True))
        return [(c.get('id', ''), c.cell_type, ''.join(c.source)) for c in nb.cells]

    def nb_add_cell(self, path, source, index=-1, cell_type='code'):
        from fastcore.nbio import read_nb, write_nb, mk_cell, dict2nb
        p = self.check(path)
        nb = read_nb(p) if p.exists() else dict2nb({'cells': [], 'metadata': {}, 'nbformat': 4, 'nbformat_minor': 5})
        cell = mk_cell(source, cell_type)
        if not cell.get('id'): cell['id'] = uuid.uuid4().hex[:8]
        nb.cells.append(cell) if index < 0 else nb.cells.insert(int(index), cell)
        p.parent.mkdir(parents=True, exist_ok=True)
        write_nb(nb, p)
        return cell['id']


    def _exec(self, code, ns):
        "Run `code` in `ns`, returning printed output plus the last expression's value."
        import contextlib, io
        buf = io.StringIO()
        tree = ast.parse(str(code))
        last = tree.body.pop() if tree.body and isinstance(tree.body[-1], ast.Expr) else None
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            if tree.body: exec(compile(tree, '<agent>', 'exec'), ns)
            value = eval(compile(ast.Expression(last.value), '<agent>', 'eval'), ns) if last else None
        out = buf.getvalue()
        if value is not None: out += ('' if not out or out.endswith('\n') else '\n') + repr(value)
        return out.strip() or '(no output)'

    def run_python(self, code):
        "Run `code` in the live namespace. Failures come back as text: a tool cannot usefully raise."
        try: return self._exec(code, self.ns)
        except Exception as e: return f'{agent_err(e)}'

    def inspect_python(self, code, scope='isolated'):
        "Run `code` against a *copy* of the namespace. Nothing the user made can move."
        if scope not in self.scopes: return f'this host only honours {self.scopes}'
        try: return self._exec(code, dict(self.ns))
        except Exception as e: return f'{agent_err(e)}'

    @property
    def scopes(self):
        "Isolated only. Overlay needs an AST policy over the real namespace, which belongs to an IDE."
        return ('isolated',)

    @property
    def kernel_kind(self): return 'inprocess'


    def run_cmd(self, command, cwd=None, timeout=120):
        """Run `command` in a shell under one of the open folders.

        Started in its own process group, and the *group* is killed on timeout. A command
        that spawns children cannot leave one behind. Stdout and stderr are interleaved.
        """
        import subprocess
        if not str(command or '').strip(): return 0, ''   # the capability probe
        if not (cwd or self._roots): raise AgentError(NO_ROOTS)
        d = self.check(cwd) if cwd else Path(self._roots[0])
        if not d.is_dir(): raise AgentError(f'not a directory: {d}')
        p = subprocess.Popen(str(command), shell=True, cwd=str(d), text=True, errors='replace',
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             start_new_session=True)
        try: out, _ = p.communicate(timeout=max(1, int(timeout)))
        except subprocess.TimeoutExpired:
            import os, signal
            try: os.killpg(p.pid, signal.SIGKILL)
            except Exception: p.kill()
            out, _ = p.communicate()
            return 124, (out or '') + f'\n[killed after {int(timeout)}s]'
        return p.returncode, out or ''

    @property
    def shell_note(self):
        return f'shell, in {self._roots[0]}' if self._roots else 'no folder to run a command in'

    def list_vars(self):
        rows = []
        for k, v in list(self.ns.items())[:MAX_VARS]:
            if k.startswith('_') or callable(v) or isinstance(v, type(ast)): continue
            try: short = repr(v)
            except Exception: short = '<unreprable>'
            rows.append(f'{k:20} {type(v).__name__:12} {short[:60]}')
        return '\n'.join(rows)

    def terminal_text(self, lines=200):
        "What this process has printed, when the application records it in `transcript`."
        return '\n'.join(str(x) for x in self.transcript[-int(lines):])


    def _fossick(self):
        if not self.web: raise NotImplementedError
        try:
            import fossick
            return fossick
        except Exception: raise NotImplementedError

    def web_search(self, query, n=20):
        "Search the web through fossick. An empty query answers `[]`: that is how `tools_for` probes."
        fossick = self._fossick()
        if not str(query).strip(): return []
        rows = fossick.search(str(query), n=int(n))   # `n` to fossick. Its own default is 10
        return [AttrDict(title=str(r.get('title', '')), url=str(r.get('href') or r.get('url', ''))) for r in rows]

    #: Below this many extracted chars, treat a 200 as an empty shell and escalate past fossick `auto`.
    THIN_PAGE = 400

    #: Dedicated fossick readers for URLs that are not HTML pages. Clone stays on `read_gh_repo`.
    READERS = (
        (re.compile(r'https?://(www\.)?github\.com/[^/]+/[^/]+/(blob|raw)/', re.I), 'read_gh_file', {}),
        (re.compile(r'https?://(www\.)?arxiv\.org/(abs|pdf)/', re.I), 'read_arxiv', dict(save_pdf=False, source=True)),
        (re.compile(r'https?://(www\.)?(youtube\.com/watch|youtu\.be/)', re.I), 'read_yt', {}),
    )

    def read_url(self, url, remember=True):
        "Page as markdown via fossick: `READERS`, then `fetch(auto=True)`, thin-page escalate, JSON-LD."
        fossick = self._fossick()
        for rx, name, kw in self.READERS:
            if not rx.search(str(url)) or (reader := getattr(fossick, name, None)) is None: continue
            try: text = _md_doc(reader(str(url), **kw))
            except Exception as e:   # a reader that cannot answer is not a URL that cannot be read
                self.note(f'{name} could not read {url} ({agent_err(e)}); fetching the page')
                break
            if text.strip(): return AttrDict(text=text, url=str(url))
            break
        page = fossick.fetch(str(url), auto=True)
        text = str(fossick.to_md(page) or '') if page is not None else ''
        if len(text.strip()) < self.THIN_PAGE:
            for opts in ({'heavy': True}, {'stealthy': True}):
                try: heavy = fossick.fetch(str(url), **opts)
                except Exception: continue
                if len((got := str(fossick.to_md(heavy) or '')).strip()) >= self.THIN_PAGE:
                    page, text = heavy, got
                    break
        if (ld := ld_json(getattr(page, 'html_content', '') or '')):
            text = f'<structured-data>\n{json.dumps(ld)[:LD_CHARS]}\n</structured-data>\n\n{text}'
        return None if not text.strip() else AttrDict(text=text, url=str(url))

    def research(self, query):
        "The cited corpus fossick assembled: its `digest`, and not the record it assembled it from."
        return str((self._fossick().research(str(query)) or {}).get('digest') or '')

    @property
    def research_note(self): return 'fossick' if self.web else 'web access is switched off'

    @property
    def approvals(self): return self._approvals

    def note(self, text):
        self.transcript.append(str(text))
        if self._note:
            try: self._note(str(text))
            except Exception: pass

A root with a little source in it:

In [ ]:
root = Path(tempfile.mkdtemp()).resolve()/'proj'
(root/'pkg').mkdir(parents=True)
(root/'pkg'/'sizes.py').write_text(
    'RESERVE = 16_384\n\n'
    'def threshold(ctx, reserve=RESERVE):\n'
    '    "Where compaction becomes due."\n'
    '    if not ctx: return None\n'
    '    return max(1, ctx - min(reserve, ctx // 4))\n\n'
    'class Budget:\n'
    '    def spend(self, n): return threshold(n)\n')
(root/'pkg'/'use.py').write_text('from .sizes import threshold\n\nprint(threshold(4096))\n')
local = LocalHost([root])
[str(p.relative_to(root)) for p in local.walk()]

['pkg/sizes.py', 'pkg/use.py']

The sandbox is the reason this class exists. `check` resolves paths before comparing them with the open folders. It therefore rejects both paths that climb out and symlinks that point out.

In [ ]:
test_fail(lambda: local.check('../../etc/passwd'), contains='outside the open folders')
(root/'escape').symlink_to('/etc')
test_fail(lambda: local.check('escape/passwd'), contains='outside the open folders')
local.check('pkg/sizes.py')

Path('/private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmp6zqcc117/proj/pkg/sizes.py')

Folder confinement is the correct default for most coding work. Some answers live in sibling checkouts, installed packages outside the environment, or logs owned by another project. `read_outside=True` permits explicit reads from those paths without permitting writes. Enumeration remains confined: `walk`, `grep` and `list_files` never leave the open folders.

In [ ]:
sibling = Path(tempfile.mkdtemp()).resolve()/'sibling'
sibling.mkdir(parents=True)
(sibling/'notes.md').write_text('RESERVE is 16_384, and it is the reply headroom.\n')

open_host = LocalHost([root], read_outside=True, index=False)
test_eq(local.read(sibling/'notes.md'), None)                     # confined: not readable at all
test_eq(open_host.read(sibling/'notes.md').split(',')[0], 'RESERVE is 16_384')
open_host.roots_note

'1 folder(s); reads may name any path on this machine, writes may not'

A write is still refused, and so are the handful of things nobody opened the sandbox in order to read. Turning it off is a decision about *source*. It is not a decision about the user's keys, which a turn would otherwise put verbatim into a cloud model's context.

In [ ]:
test_fail(lambda: open_host.check(sibling/'notes.md'), contains='outside the open folders')
test_fail(lambda: open_host.check(Path.home()/'.ssh'/'id_rsa', reading=True), contains='credentials')
test_eq(denied('/home/k/.aws/credentials'), True)
test_eq(denied(root/'pkg'/'sizes.py'), False)
test_fail(lambda: open_host.write(sibling/'notes.md', 'no'), contains='outside the open folders')

A relative path is taken against the first open folder, and a missing file can be demanded here rather than discovered three calls later.

In [ ]:
test_fail(lambda: local.check('pkg/nope.py', must_exist=True), contains='no such file')
local.read('pkg/sizes.py').splitlines()[0]

'RESERVE = 16_384'

Construction starts `Kosha.sync` in a daemon thread for every open root. It starts before the model does so indexing overlaps model startup. An existing `.kosha` is incremental and is normally ready before the first search.

In [ ]:
local.wait_index(120), local.search_note

(True,
 'Kosha semantic + keyword index over 1 folder(s) and environment fused with ripgrep')

Kosha combines keyword and semantic retrieval in one `context` call across the repository, environment and call graph. `search` fuses those results with ripgrep through `litesearch.rrf_all`, the same rank fusion used by `Vault.federate`. A rename query therefore retains both semantic neighbours and exact call sites.

In [ ]:
hits = local.search('where compaction becomes due')
[(h.path, h.line, h.symbol) for h in hits[:3]]

[('/Users/71293/code/personal/orgs/leela/leela/agent/compact.py',
  129,
  'leela.agent.compact.threshold'),
 ('/private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmp6zqcc117/proj/pkg/sizes.py',
  4,
  ''),
 ('/Users/71293/code/personal/orgs/leela/.venv/lib/python3.13/site-packages/llmsurgery/compact.py',
  35,
  'llmsurgery.compact.compact_enc')]

In [ ]:
assert hits, 'search returned nothing'
# Environment hits may rank ahead of the temporary project. It must still be returned.
size_hit = next((h for h in hits if Path(h.path) == root/'pkg'/'sizes.py'), None)
assert size_hit is not None, 'search did not return sizes.py'
if size_hit.symbol: test_eq(size_hit.symbol.endswith('threshold'), True)
local.index_ready

AssertionError: ==:
/Users/71293/code/personal/orgs/leela/leela/agent/compact.py
/private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmp6zqcc117/proj/pkg/compact.py

`public_api` asks the index the one question a source file cannot answer: the whole exported surface of a package, the `@patch`-added methods included. With no index it raises rather than returning `[]`. "this package exports nothing" and "nothing could be looked up" must not reach the model as the same answer.

In [ ]:
api = local.public_api('fastcore')
assert api, 'no public API for fastcore'
test_eq(any(h.symbol == 'fastcore.basics.store_attr' for h in api), True)
test_eq(local.public_api('nosuchpkg_ramabana'), [])                 # installed, and exporting nothing
test_fail(lambda: LocalHost([root], index=False).public_api('fastcore'), contains='no code index')
api[0]

While the first sync is still running, search still answers from the ripgrep leg rather than making the model wait. `search_note` always says which engines are live. "no matches" can be told apart from "the semantic index was not ready".

In [ ]:
test_eq(local.web_search(''), [])          # the capability probe must not hit the network
local.research_note

'fossick'

In [ ]:
local.search('threshold')[:2], local.search_note

([/private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpf71bmso3/proj/pkg/sizes.py:9  sizes.Budget.spend  def spend(self, n): return threshold(n),
  /private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpf71bmso3/proj/pkg/sizes.py:3  sizes.threshold  def threshold(ctx, reserve=RESERVE): "Where compaction becomes due." if not ctx: return None return max(1, ctx - min(reserve, ctx // 4))],
 'Kosha semantic + keyword index over 1 folder(s) and environment')

Symbols come from parsing rather than grepping. A method is reported at its own depth and a `def` inside a docstring is not reported at all.

In [ ]:
[(h.symbol, h.line, h.score) for h in local.symbols('pkg/sizes.py')]

[('threshold', 3, 0), ('Budget', 8, 0), ('Budget.spend', 9, 1)]

`peers` finds mentions of the symbol defined at a given line. It provides the useful call-site half of semantic peer search even when the host has no semantic index.

In [ ]:
local.peers('pkg/sizes.py', 3)

[/private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpf71bmso3/proj/pkg/sizes.py:9  sizes.Budget.spend  def spend(self, n): return threshold(n),
 /private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpf71bmso3/proj/pkg/sizes.py:8  sizes.Budget  class Budget: def spend(self, n): return threshold(n),
 /Users/71293/code/personal/orgs/leela/.venv/lib/python3.13/site-packages/networkx/algorithms/threshold.py:826  networkx.algorithms.threshold.random_threshold_sequence  def random_threshold_sequence(n, p, seed=None): """ Create a random threshold sequence of size n. A creation sequence is built by randomly choosing d's with probability p and i's with probability 1-p. s=nx.random_threshold_sequence(10,0.5) ,
 /Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/spacy/cli/find_threshold.py:27  spacy.cli.find_threshold.find_threshold_cli  def find_threshold_cli( # fmt: off model: str = Arg(..., help="Model name or path"), data_path: Path = Arg( ..., help="Locatio

`text_at` is what `Agent.changes()` diffs, and it is notebook-aware: a notebook diffs as its cell sources rather than as nbformat JSON. A file that does not exist yet is `''`. Creating one diffs as a pure addition instead of as an error.

In [ ]:
cid = local.nb_add_cell('nb/demo.ipynb', 'x = threshold(4096)\nx')
local.nb_cells('nb/demo.ipynb'), local.text_at('nb/demo.ipynb')

([('ac8d20e4', 'code', 'x = threshold(4096)\nx')], 'x = threshold(4096)\nx')

In [ ]:
test_eq(local.text_at('pkg/nope.py'), '')
cid

'ac8d20e4'

The live namespace persists between calls. This persistence gives the "bind results to new names" contract its effect.

In [ ]:
local.run_python('import math\nradii = [1, 2, 3]'), local.run_python('areas = [math.pi*r*r for r in radii]\nareas[:2]')

('(no output)', '[3.141592653589793, 12.566370614359172]')

A trailing expression returns its value. The tool can therefore inspect state without requiring `print`.

In [ ]:
local.run_python('len(areas)'), local.run_python('print("a side effect")')

('3', 'a side effect')

Failures come back as text. A tool that raises ends a turn, and a misspelled name is not worth ending a turn over.

In [ ]:
local.run_python('no_such_name + 1')

"NameError: name 'no_such_name' is not defined"

`inspect_python` runs against a *copy*. Nothing done there can move what the user made. This host advertises only the isolated scope: an overlay needs an AST policy over the real namespace, which is an IDE's job.

In [ ]:
local.inspect_python('radii.append(99)\nlen(radii)'), local.run_python('len(radii)')

('4', '4')

In [ ]:
test_eq(local.scopes, ('isolated',))
print(local.list_vars())

radii                list         [1, 2, 3, 99]
areas                list         [3.141592653589793, 12.566370614359172, 28.274333882308138]


## Skills

A skill is know-how the agent can read on demand: a package that documents itself, or a `SKILL.md` file in a repository. Only names and one-line descriptions go in the system prompt. The bodies are fetched by the `read_skill` tool, which is what keeps a dozen skills affordable.

In [ ]:
#| export
GROUP,EXTRA_MODULES,MAX_SKILL_CHARS = 'pyskills',('exhash.skill',),20_000

def _describe(text, mx=300):
    'A one-line description from a skill body: its first paragraph, collapsed.'
    body = (text or '').strip()
    if not body: return ''
    para = body.split('\n\n', 1)[0]
    one = ' '.join(para.split())
    return one if len(one) <= mx else one[:mx - 1].rstrip() + '…'

@dataclass
class Skill:
    'One skill: how to name it, when it applies, and how to get the whole text.'
    name: str
    source: str                   # 'pyskill' | 'md'
    description: str = ''
    where: str = ''               # module path or file path, shown so a person can go read it
    _text: object = field(default=None, repr=False)

    def text(self):
        'The full skill body, clipped. Never raises: a broken skill reports itself as one.'
        try: t = self._text() if callable(self._text) else (self._text or '')
        except Exception as e: return f'could not read skill {self.name}: {agent_err(e)}'
        t = str(t)
        return t if len(t) <= MAX_SKILL_CHARS else t[:MAX_SKILL_CHARS] + f'\n…[{len(t)-MAX_SKILL_CHARS} more chars]'

    def dict(self): return {'name': self.name, 'source': self.source,'description': self.description, 'where': self.where}

A description is the first paragraph, collapsed to one line. The same convention a `SKILL.md` frontmatter `description` follows. A package and a file read alike in the index.

In [ ]:
_describe('''Search the web and read results into durable memory.

The rest of this document explains the memory layout, which the index does not need.''')

'Search the web and read results into durable memory.'

A `Skill` holds a *way to get* the body rather than the body. Discovering forty skills does not read forty files. The body is clipped, and a loader that raises reports itself as the skill's text instead of taking the turn down with it.

In [ ]:
s = Skill('nbdev', 'md', 'Develop nbdev projects.', 'nbs/SKILL.md', _text=lambda: 'the whole skill body')
s.text(), s.dict()

('the whole skill body',
 {'name': 'nbdev',
  'source': 'md',
  'description': 'Develop nbdev projects.',
  'where': 'nbs/SKILL.md'})

In [ ]:
def _explodes(): raise FileNotFoundError('SKILL.md')
Skill('broken', 'md', _text=_explodes).text()

'could not read skill broken: FileNotFoundError: SKILL.md'

Skills come from installed packages (the `pyskills` entry-point group) and from `<name>/SKILL.md` directories. The precedence is deliberate: a file beats a package, because the package's skill is the general advice and the one in your own repository is the correction.

In [ ]:
#| export
def _mod_skill(name, modpath):
    "A `Skill` for a module, without importing it until someone asks for the body."
    def load():
        from importlib import import_module
        return import_module(modpath).__doc__ or ''
    # the description needs the docstring, and there is no way to read one without importing
    try:
        from importlib import import_module
        doc = import_module(modpath).__doc__ or ''
    except Exception: return None
    if not doc.strip(): return None
    return Skill(name=name, source='pyskill', description=_describe(doc), where=modpath, _text=load)

def _pyskills():
    "Every module published under the `pyskills` entry-point group, plus the known stragglers."
    out, seen = [], set()
    try:
        from importlib.metadata import entry_points
        eps = list(entry_points(group=GROUP))
    except Exception:
        eps = []
    for ep in eps:
        mod = getattr(ep, 'value', None) or ep.name
        if mod in seen: continue
        seen.add(mod)
        if (s := _mod_skill(ep.name.split('.')[-1] or ep.name, mod)): out.append(s)
    for mod in EXTRA_MODULES:
        if mod in seen: continue
        seen.add(mod)
        if (s := _mod_skill(mod.split('.')[0], mod)): out.append(s)
    return out

def skill_dirs(roots=(), cfg=None):
    "Where SKILL.md files are looked for, in increasing precedence: user first. A project can override."
    from pathlib import Path
    ds = []
    if cfg is not None: ds.append(Path(cfg)/'skills')
    ds.append(Path.home()/'.agents'/'skills')
    for r in roots: ds += [Path(r)/'.leela'/'skills', Path(r)/'.agents'/'skills']
    return ds

def _md_skills(d):
    "Skills in one directory, following the Agent Skills layout: `<name>/SKILL.md`."
    from pathlib import Path
    d = Path(d)
    if not d.is_dir(): return []
    out = []
    for p in sorted(d.iterdir()):
        if not p.is_dir(): continue
        f = p/'SKILL.md'
        if not f.exists(): continue
        try: raw = f.read_text(encoding='utf-8')
        except Exception: continue
        meta, body = frontmatter(raw)
        out.append(Skill(name=meta.get('name') or p.name, source='md',
                         description=meta.get('description') or _describe(body),
                         where=str(f), _text=body))
    return out

def discover(roots=(), cfg=None, extra=()):
    "Every skill available to this agent. Pyskills, then `skill_dirs`, then `extra`. Later winning."
    by_name = {}
    for s in _pyskills(): by_name[s.name] = s
    for d in skill_dirs(roots, cfg):
        for s in _md_skills(d): by_name[s.name] = s
    for s in extra or (): by_name[s.name] = s
    return sorted(by_name.values(), key=lambda s: s.name)

The search path, in increasing precedence. Personal habits first. A project can override them:

In [ ]:
[str(p) for p in skill_dirs(roots=['/proj'], cfg='/home/k/.config/leela')]

['/home/k/.config/leela/skills',
 '/Users/71293/.agents/skills',
 '/proj/.leela/skills',
 '/proj/.agents/skills']

Written out, a skill directory follows the Agent Skills layout, and its frontmatter supplies the name and description:

In [ ]:
tmp = Path(tempfile.mkdtemp())
d = tmp/'.agents'/'skills'/'notebook-tests'
d.mkdir(parents=True)
(d/'SKILL.md').write_text('---\nname: notebook-tests\ndescription: Write tests as notebook cells.\n---\n\nUse `test_eq`.\n')
[(x.name, x.source, x.description) for x in _md_skills(tmp/'.agents'/'skills')]

[('notebook-tests', 'md', 'Write tests as notebook cells.')]

`discover` merges every source into one list, with later sources winning on a name clash.

In [ ]:
skills = discover(roots=[tmp])
[x.name for x in skills]

['coding_patterns',
 'design-taste-frontend',
 'editskill',
 'exhash',
 'fastcdp',
 'find-skills',
 'full-output-enforcement',
 'ghapi',
 'hf-cli',
 'high-end-visual-design',
 'industrial-brutalist-ui',
 'kosha',
 'minimalist-ui',
 'nbdev',
 'notebook-tests',
 'read_md',
 'redesign-existing-projects',
 'rgapi',
 'skill',
 'stitch-design-taste',
 'vishalakshi']

`skill_index` is what goes in the system prompt, and `find` is how `read_skill` resolves what the model asked for.

In [ ]:
#| export
SKILL_DESC_MAX = 160   # per skill. One verbose description cannot crowd out the rest

def _clip_desc(s, n=SKILL_DESC_MAX):
    "One line, clipped at a word boundary: in the index a description only has to be pickable."
    s = ' '.join(str(s).split())
    if len(s) <= n: return s
    cut = s.rfind(' ', 0, n)
    return s[:cut if cut > 0 else n].rstrip(' .,;:\u2014-') + '\u2026'

def skill_index(skills):
    "The block that goes in the system prompt: names and clipped descriptions, never bodies."
    if not skills: return ''
    rows = '\n'.join(f'- `{s.name}` -- {_clip_desc(s.description)}' for s in skills)
    return ('\n\n## Skills\n\nKnow-how available to you. Read one with `read_skill(name)` when its '
            'description matches what you are about to do, *before* you do it -- several of these '
            'describe tools already installed in this environment, so the code they discuss is '
            'also searchable with `search_code`.\n\n' + rows)

def find(skills, name):
    "A skill by exact name, then unique prefix, then unique substring. Ambiguity is None, not a guess."
    if not name: return None
    n = name.strip().lower()
    if (exact := [s for s in skills if s.name.lower() == n]): return exact[0]
    for pred in (lambda s: s.name.lower().startswith(n), lambda s: n in s.name.lower()):
        if len(hits := [s for s in skills if pred(s)]) == 1: return hits[0]
    return None

In [ ]:
print(skill_index([s for s in skills if s.name == 'notebook-tests']))



## Skills

Know-how available to you. Read one with `read_skill(name)` when its description matches what you are about to do, *before* you do it -- several of these describe tools already installed in this environment, so the code they discuss is also searchable with `search_code`.

- `notebook-tests` -- Write tests as notebook cells.


In [ ]:
from fastcore.test import test_eq

# A description written to be *found* runs to a thousand characters. In the index it is one line.
wordy = Skill('wordy', 'md', description=' '.join(['trigger']*200))
row = skill_index([wordy]).splitlines()[-1]
assert len(row) < SKILL_DESC_MAX + 20, len(row)
assert row.endswith('\u2026')
test_eq(_clip_desc('short enough'), 'short enough')
test_eq(_clip_desc('  collapsed   whitespace  '), 'collapsed whitespace')

A name resolves exactly, then by unique prefix, then by unique substring. Ambiguity returns `None` rather than a guess: a model that asked for `note` and silently got the wrong skill will read the wrong reference and then confidently do the wrong thing.

In [ ]:
find(skills, 'notebook-tests'), find(skills, 'notebook'), find(skills, 'nope')

(Skill(name='notebook-tests', source='md', description='Write tests as notebook cells.', where='/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmp1hnmtoaq/.agents/skills/notebook-tests/SKILL.md'),
 Skill(name='notebook-tests', source='md', description='Write tests as notebook cells.', where='/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmp1hnmtoaq/.agents/skills/notebook-tests/SKILL.md'),
 None)

In [ ]:
two = [Skill('edit', 'md'), Skill('editor', 'md')]
test_eq(find(two, 'edit').name, 'edit')        # exact wins over prefix
find(two, 'edi') is None                        # ambiguous: say so

True

## Extensions

An extension is a Python file the user drops in a directory. `Registry` is everything it is handed: tools, skills, slash commands, lifecycle hooks and the approval policy. What is deliberately absent is any route to a backend's internals. An extension that pokes at a litert conversation would break on the next model switch, and break silently.

In [ ]:
#| export
EVENTS = ('before_turn', 'after_turn', 'before_tool', 'after_tool', 'compact', 'approval')

In [ ]:
#| export
class Registry:
    "What `setup(ext)` is handed: everything an extension may add, and no route to a backend's internals."

    def __init__(self, host=None, agent=None):
        self.host, self.agent = host, agent
        self.tools, self.skills, self.commands = [], [], {}
        self.hooks = {e: [] for e in EVENTS}
        self.approve = None
        self.notes = []          # one line per extension: loaded, or why not


    def tool(self, f):
        "Add a tool, as a decorator: a plain function with type hints and the docstring the model reads."
        self.tools.append(f)
        return f

    def skill(self, name, text, description=''):
        "Add a skill the discovery pass would not find. A file, a string, anything callable."
        s = Skill(name=name, source='ext', description=description or _describe(text if isinstance(text, str) else ''),
                  where='extension', _text=text)
        self.skills.append(s)
        return s

    def command(self, name, fn, help=''):
        "Add a slash command. `fn(agent, arg)` returns text for the frontend to show."
        self.commands[name.lstrip('/')] = (fn, help)
        return fn

    def on(self, event, fn):
        "Hook a harness lifecycle event. Unknown event names are an error, not a silent no-op."
        if event not in EVENTS: raise KeyError(f'unknown event {event!r}; known: {", ".join(EVENTS)}')
        self.hooks[event].append(fn)
        return fn

    def approval(self, fn):
        "Replace the approval policy wholesale. The last extension to call this wins."
        self.approve = fn
        return fn


    def fire(self, event, *args, **kw):
        "Run every hook for `event`, swallowing failures. Returns how many ran cleanly."
        n = 0
        for f in self.hooks.get(event, ()):
            try: f(*args, **kw); n += 1
            except Exception as e: self.notes.append(f'{event} hook failed: {agent_err(e)}')
        return n

In [ ]:
#| export
def ext_dirs(roots=(), cfg=None, project=False):
    "Where extensions are looked for. Project directories only when explicitly allowed."
    ds = []
    if cfg is not None: ds.append(Path(cfg)/'extensions')
    if project:
        for r in roots: ds.append(Path(r)/'.leela'/'extensions')
    return ds


def load(reg, roots=(), cfg=None, project=False, paths=()):
    "Run every extension found, calling its `setup(reg)`. A file with no `setup` is left alone."
    files = []
    for d in ext_dirs(roots, cfg, project):
        if Path(d).is_dir(): files += sorted(p for p in Path(d).glob('*.py') if not p.name.startswith('_'))
    for p in paths or ():
        p = Path(p)
        files += sorted(p.glob('*.py')) if p.is_dir() else [p]
    for f in files:
        try: ns = runpy.run_path(str(f))
        except Exception as e:
            reg.notes.append(f'{f.name}: failed to load ({agent_err(e)})')
            continue
        fn = ns.get('setup')
        if not callable(fn):
            reg.notes.append(f'{f.name}: loaded, no setup()')
            continue
        before = (len(reg.tools), len(reg.skills), len(reg.commands))
        try: fn(reg)
        except Exception as e:
            reg.notes.append(f'{f.name}: setup() failed ({agent_err(e)})')
            continue
        d = [n - b for n, b in zip((len(reg.tools), len(reg.skills), len(reg.commands)), before)]
        reg.notes.append(f'{f.name}: {d[0]} tool(s), {d[1]} skill(s), {d[2]} command(s)')
    return reg

An extension is a file with a `setup(reg)`. This one adds a tool and a command:

In [ ]:
extdir = tmp/'extensions'
extdir.mkdir()
(extdir/'wordcount.py').write_text('''
def setup(reg):
    @reg.tool
    def word_count(path: str) -> str:
        "Count the words in a file in the open folders."
        return str(len((reg.host.read(path) or "").split()))

    reg.command("wc", lambda agent, arg: "counted", help="count words")
''')
reg = load(Registry(host=MemHost({'/proj/a.py': 'def a(): return 1\n'})), paths=[extdir])
reg.notes

['wordcount.py: 1 tool(s), 0 skill(s), 1 command(s)']

The registered tool is an ordinary function, with the docstring the model will read, and it works against the host it was given.

In [ ]:
reg.tools[0]('/proj/a.py'), list(reg.commands)

('4', ['wc'])

A file without `setup` loads without registering anything. This permits shared helper modules in the extension directory. A file that raises is reported and skipped.

In [ ]:
(extdir/'helpers.py').write_text('SHARED = 1\n')
(extdir/'broken.py').write_text('raise RuntimeError("bad import")\n')
load(Registry(), paths=[extdir]).notes

['broken.py: failed to load (RuntimeError: bad import)',
 'helpers.py: loaded, no setup()',
 'wordcount.py: 1 tool(s), 0 skill(s), 1 command(s)']

Hooking an event that does not exist is an error rather than a silent no-op, because a misspelled hook that never fires is the hardest kind of extension bug to see.

In [ ]:
test_fail(lambda: reg.on('before_lunch', print), contains='unknown event')
EVENTS

('before_turn',
 'after_turn',
 'before_tool',
 'after_tool',
 'compact',
 'approval')

Firing is fail-soft in the other direction: a hook that raises is recorded and the turn continues, since an extension should not be able to end a session.

In [ ]:
reg.on('before_turn', lambda **kw: 1/0)
reg.fire('before_turn'), reg.notes[-1]

(0, 'before_turn hook failed: ZeroDivisionError: division by zero')

## Tool plumbing

Every tool shares three pieces of plumbing. Clipping protects the context window. `_cmds` parses hash-verified edit commands from JSON. Capability probes determine which tools a host can support. `WRITE_TOOLS` identifies calls that require an approval policy.

In [ ]:
#| export
MAX_TOOL_CHARS = 6000   # chars per tool result, budgeted for the smallest model. See `Agent(tool_max_len=)`
MAX_HITS = 20

# Rehearsing a merge is not approving one. The git tools split before `WRITE_TOOLS` uses them.
GIT_READ_TOOLS = ('git_status', 'git_divergence', 'git_rebase_preview')
GIT_WRITE_TOOLS = frozenset({'git_remote', 'git_checkout'})
GIT_TOOLS = (*GIT_READ_TOOLS, *sorted(GIT_WRITE_TOOLS))

# The tools that change something on disk, in the live session, or on the machine. See `Approvals`.
WRITE_TOOLS = frozenset({'edit_file', 'replace_text', 'create_file', 'edit_cell', 'add_cell',
                         'run_python', 'run_shell', 'memory_forget', 'create_skill',
                         'cancel_watch', 'cart_add', 'cart_remove', 'add_root'}) | GIT_WRITE_TOOLS

# Every tool failure starts with this: no engine carries an `is_error` flag. The flag is in the text.
ERR = 'ERROR: '


def err(what, e=None):
    "One tool failure, spelled the way every other tool spells it."
    return f'{ERR}{what}' + (f': {agent_err(e)}' if e is not None else '')


def failed(result):
    "Whether a tool result is a failure. The one place that knows how a failure is spelled."
    return str(result or '').startswith(ERR)


def clip(s, n=MAX_TOOL_CHARS, more=''):
    "Truncate a tool result to `n` chars. A caller with a way to resume passes it as `more`."
    s = str(s)
    if len(s) <= n: return s
    cut = s[:n]
    nl = cut.rfind('\n')                    # never end mid-line: the line would look complete
    if nl > n * 0.6: cut = cut[:nl]
    note = f'[truncated: {len(cut)} of {len(s)} chars shown'
    return cut + f'\n…{note}. {more}]' if more else cut + f'\n…{note}]'


def clip_lines(lines, start=1, n=MAX_TOOL_CHARS, more='', empty='(nothing)'):
    """Render `lines` within the budget, and say which line to resume from.

    One line longer than the whole budget is cut by characters instead, and the notice says
    characters. The model is not invited to resume at the same too-long line.
    """
    lines = list(lines)
    if not lines: return empty
    out, used = [], 0
    for i, line in enumerate(lines):
        line = str(line)
        if used + len(line) + 1 > n:
            if out:
                rest = len(lines) - i
                tail = f'\n…[{rest} more line(s) not shown'
                hint = more.format(next=start + i) if '{next}' in more else more
                return '\n'.join(out) + (f'{tail}. {hint}]' if hint else f'{tail}]')
            keep = max(1, n - 1)
            rest = len(lines) - 1
            more_lines = f', and {rest} more line(s) not shown' if rest else ''
            return line[:keep] + f'\n…[line {start} is {len(line)} chars; {keep} shown{more_lines}]'
        out.append(line); used += len(line) + 1
    return '\n'.join(out)


def _cmds(commands):
    "Models emit JSON and exhash wants tuples, nested ones included: `[[...]]` becomes `[(...)]`."
    if isinstance(commands, str): commands = json.loads(commands)
    if not isinstance(commands, list): raise ValueError('commands must be a JSON list of command arrays')
    def _t(c):
        if not isinstance(c, (list, tuple)): raise ValueError(f'each command must be an array, got {type(c).__name__}')
        return tuple(_t(x) if isinstance(x, (list, tuple)) else x for x in c)
    return [_t(c) for c in commands]


@functools.lru_cache(maxsize=None)
def _takes_reading(cls):
    "Whether this host's `check` understands the read-only flag. Asked once per class."
    import inspect
    try: return 'reading' in inspect.signature(cls.check).parameters
    except (TypeError, ValueError): return False


def readable(host, path, must_exist=False):
    "Resolve a path a tool is only going to read. A host that predates the `reading` flag never sees it."
    if _takes_reading(type(host)): return host.check(path, must_exist=must_exist, reading=True)
    return host.check(path, must_exist=must_exist)


def _declared(host, group):
    "What `host.capabilities` says about `group`, or None when it does not say."
    try: d = host.capabilities or {}
    except Exception: return None
    return bool(d[group]) if group in d else None


def _probe(host, *calls):
    "Whether every one of `calls` is supported. A host says 'no' by raising `NotImplementedError`."
    for f in calls:
        try: f()
        except NotImplementedError: return False
        except Exception: pass
    return True


def _supports(host, name, probe=None):
    """Whether `host` implements `name`, by asking whether it overrode the method.

    For a capability with no harmless probe. `probe` is the *contracted* harmless call, and it
    catches a host that overrides the method and then refuses anyway.
    """
    own, base = getattr(type(host), name, None), getattr(Host, name, None)
    if own is None or own is base: return False
    if probe is None: return True
    try: probe()
    except NotImplementedError: return False
    except Exception: pass
    return True


def _has(host, group, *calls):
    "Whether `host` supports `group`: its own declaration when it makes one, a harmless call otherwise."
    d = _declared(host, group)
    return _probe(host, *calls) if d is None else d

`readable` is how every read-only tool resolves a path. It exists so the flag is a host capability rather than a tool assumption: a host written before `reading` existed is sent the call it has always been sent, and simply never answers for anything outside its folders.

In [ ]:
class OldHost(NullHost):
    "A host from before the flag: `check` takes two arguments and always will."
    def check(self, path, must_exist=False): return Path(path)

test_eq(_takes_reading(OldHost), False)
test_eq(_takes_reading(LocalHost), True)
test_eq(str(readable(OldHost(['/proj']), '/anywhere/x.py')), '/anywhere/x.py')
test_eq(str(readable(open_host, sibling/'notes.md')), str(sibling/'notes.md'))

In [ ]:
clip('the whole file, all of it', 12)

'the whole fi\n…[truncated: 12 of 25 chars shown]'

Models emit JSON and exhash wants tuples. `_cmds` converts, recursing into the nested command arrays that `g`/`v` take.

In [ ]:
_cmds('[["12|a1b2|", "s", "old", "new"], ["30|9f3c|", "a", "appended"]]')

[('12|a1b2|', 's', 'old', 'new'), ('30|9f3c|', 'a', 'appended')]

In [ ]:
test_fail(lambda: _cmds('{"not": "a list"}'), contains='must be a JSON list')
sorted(WRITE_TOOLS)

['add_cell',
 'cancel_watch',
 'cart_add',
 'cart_remove',
 'create_file',
 'create_skill',
 'edit_cell',
 'edit_file',
 'memory_forget',
 'replace_text',
 'run_python',
 'run_shell']

## Seeing the code

The first group any host gets: search the index, find code shaped like a given function, outline a file, list the files. Every result names the exact path and how to address it, because a model that has to guess whether something is a notebook will guess wrong.

In [ ]:
#| export
def code_tools(host, mx=MAX_TOOL_CHARS):
    "Seeing the code: the index, the shapes in it, and the files it covers."

    def search_code(query: str) -> str:
        """Search the codebase and every installed package for `query`.

        Semantic when the code index is built, a literal scan otherwise. Use this before
        writing anything non-trivial: the answer is usually already in the environment.
        """
        hits = host.search(query, limit=MAX_HITS)
        if not hits: return f'no matches ({host.search_note})'
        rows = []
        for h in hits:
            target = ('NOTEBOOK -- use this exact path with notebook_cells, then view_cell/edit_cell'
                      if str(h.path).lower().endswith('.ipynb')
                      else 'FILE -- use this exact path with view_file/edit_file')
            rows.append(f'{h.path}:{h.line}  {h.symbol or ""}  {h.text}\n  {target}')
        return clip(f'[{host.search_note}]\n' + '\n'.join(rows), mx)

    def similar_code(path: str, line: int = 1) -> str:
        "Find code shaped like the function at `path`:`line`. Every place a pattern was already used."
        hits = host.peers(str(readable(host, path)), int(line), limit=MAX_HITS)
        if not hits: return f'nothing similar ({host.search_note})'
        return clip('\n'.join(f'{h.path}:{h.line}  {h.symbol or ""}  {h.text}' for h in hits), mx)

    def outline(path: str) -> str:
        "The defs and classes in one file, with line numbers."
        syms = host.symbols(str(readable(host, path)))
        if not syms: return f'no symbols in {path}'
        return clip('\n'.join(f'{int(getattr(s, "score", 0))*" "}{s.line}: {s.symbol}' for s in syms), mx)

    def list_files(pattern: str = '') -> str:
        "Files in the open folders, optionally filtered by a substring of the path."
        ps = [str(p) for p in host.walk()]
        if pattern: ps = [p for p in ps if pattern.lower() in p.lower()]
        return clip_lines(ps, n=mx, more='narrow `pattern`', empty='no matching files')

    def grep(pattern: str, path_filter: str = '', regex: bool = True, ignore_case: bool = False) -> str:
        """Find every line in the open folders matching `pattern`, exactly.

        The literal counterpart to `search_code`, and not a replacement for it. Use
        `search_code` for "how does this work". It is a semantic index and it covers
        installed packages. Use `grep` when you know the string: a symbol you are about to
        rename, an error message, an import, a call site you must not miss. An index answers
        with what is *like* the query. This answers with what *is* the query, which is what
        a rename or an audit needs.

        `path_filter` is a substring of the path (`tests/`, `.py`). Set `regex=False` to
        match `pattern` literally when it contains regex punctuation.
        """
        if not str(pattern or '').strip(): return err('grep needs a pattern')
        flags = re.IGNORECASE if ignore_case else 0
        try: rx = re.compile(pattern if regex else re.escape(pattern), flags)
        except re.error as e: return err('bad pattern', e)
        # ask the host first: it may have ripgrep, and reading every file costs a `check` each
        try: fast = host.grep(pattern, path_filter=path_filter, regex=regex,
                              ignore_case=ignore_case, limit=MAX_GREP_HITS)
        except Exception: fast = None
        if fast is not None:
            if not fast: return f'no matches for {pattern!r}'
            capped = len(fast) >= MAX_GREP_HITS
            head = f'{len(fast)}{"+" if capped else ""} match(es)'
            rows = [f'{h.path}:{h.line}: {h.text}' for h in fast]
            return clip_lines([head] + rows, n=mx, more='narrow `pattern` or set `path_filter`')
        pf, hits, scanned, capped = str(path_filter or '').lower(), [], 0, False
        for p in host.walk():
            sp = str(p)
            if pf and pf not in sp.lower(): continue
            try: text = host.read(sp)
            except Exception: continue
            if not text: continue
            scanned += 1
            for i, line in enumerate(text.splitlines(), 1):
                if rx.search(line):
                    hits.append(f'{sp}:{i}: {line.strip()[:200]}')
                    if len(hits) >= MAX_GREP_HITS: capped = True; break
            if capped: break
        if not hits: return f'no matches for {pattern!r} in {scanned} file(s)'
        head = f'{len(hits)}{"+" if capped else ""} match(es) in {scanned} file(s) searched'
        return clip_lines([head] + hits, n=mx, more='narrow `pattern` or set `path_filter`')

    def ls(path: str = '') -> str:
        """List one directory: its subdirectories, then its files with sizes.

        For finding your way around. `list_files` walks everything and `grep` reads
        everything. This just says what is here, which is usually the cheaper question.
        Empty `path` lists each open folder.
        """
        roots = ([readable(host, path)] if str(path or '').strip()
                 else [host.check(r) for r in host.roots])
        out = []
        for d in roots:
            if not d.exists(): out.append(f'{d}: does not exist'); continue
            if d.is_file(): out.append(f'{d}  ({d.stat().st_size} bytes, a file)'); continue
            try: kids = sorted(d.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
            except Exception as e: out.append(err(f'cannot list {d}', e)); continue
            out.append(f'{d}/')
            for k in kids:
                if k.name.startswith('.') and k.name not in ('.agents', '.leela'): continue
                try: out.append(f'  {k.name}/' if k.is_dir() else f'  {k.name}  {k.stat().st_size}')
                except Exception: out.append(f'  {k.name}')
        return clip_lines(out, n=mx, more='name a subdirectory to list it', empty='(nothing)')

    def public_api(package: str) -> str:
        """Every public name a package exports, with its docstring and where it is defined.

        The whole surface of `package` in one call, `@patch`-added methods included, which
        reading one source file will not show. Ask before writing against an unfamiliar
        package: what you were about to implement is often already exported.
        """
        if not str(package or '').strip(): return err('public_api needs a package name')
        try: api = host.public_api(package)
        except Exception as e: return err(f'cannot list the API of {package}', e)
        if not api:
            # the index answered. This is not a missing index. Say what it does not cover
            return (f'nothing public indexed for {package!r}. The index covers installed packages and '
                    f'the open folders, not the stdlib')
        rows = [f'{h.symbol}  {h.path}:{h.line}  {h.text}' for h in api]
        return clip_lines([f'{len(rows)} public name(s) in {package}'] + rows, n=mx, more='read one with view_file')

    tools = [search_code, grep, ls, similar_code, outline, list_files]
    if _probe(host, lambda: host.public_api('')): tools.append(public_api)
    return tools

In [ ]:
host = MemHost({'/proj/a.py': 'def a(): return 1\n', '/proj/b.py': 'def b(): return a() + 1\n'})
search_code, grep, ls, similar_code, outline, list_files = code_tools(host)
print(search_code('return'))

[memory]
/proj/a.py:1    def a(): return 1
  FILE -- use this exact path with view_file/edit_file
/proj/b.py:1    def b(): return a() + 1
  FILE -- use this exact path with view_file/edit_file


When nothing matches, the answer says which engine answered. "no matches" can be told apart from "no index".

In [ ]:
search_code('nonexistent'), list_files('b.py')

('no matches (memory)', '/proj/b.py')

The group is only the tools this host can answer: `MemHost` has no index. It is offered no `public_api` rather than one that always fails.

In [ ]:
test_eq('public_api' in [t.__name__ for t in code_tools(host)], False)
api_tool = {t.__name__: t for t in code_tools(local)}['public_api']
test_eq(failed(api_tool('')), True)
print('\n'.join(api_tool('fastcore').splitlines()[:3]))

## Files

Files are read and written by hash-verified address. `view_file` returns `lineno|hash|content` lines, and those hashes are the addresses `edit_file` takes. The view is also the address book, and an edit built on a stale view fails instead of damaging the wrong line.

In [ ]:
#| export
def _edits(edits):
    "A JSON string, `{'oldText','newText'}` dicts, or `[old, new]` pairs: all three are unambiguous."
    if isinstance(edits, str): edits = json.loads(edits)
    if isinstance(edits, dict): edits = [edits]
    if not isinstance(edits, (list, tuple)): raise ValueError('edits must be a JSON array')
    out = []
    for e in edits:
        if isinstance(e, dict):
            if 'oldText' not in e or 'newText' not in e:
                raise ValueError("each edit needs 'oldText' and 'newText'")
            out.append((str(e['oldText']), str(e['newText'])))
        elif isinstance(e, (list, tuple)) and len(e) == 2: out.append((str(e[0]), str(e[1])))
        else: raise ValueError('each edit must be {"oldText":…,"newText":…} or [old, new]')
    return out


def _apply_edits(text, edits):
    "Apply exact-text edits to `text`, or raise saying which one is wrong and why."
    spans = []
    for i, (old, new) in enumerate(edits, 1):
        if not old: raise ValueError(f'edit {i}: oldText is empty; use create_file to write a whole file')
        n = text.count(old)
        if n == 0:
            raise ValueError(f'edit {i}: oldText not found. It must match the file exactly, '
                             f'including indentation. Re-read the file and try again')
        if n > 1:
            raise ValueError(f'edit {i}: oldText matches {n} places. Include more surrounding '
                             f'lines so it matches exactly one')
        at = text.index(old)
        spans.append((at, at + len(old), new, i))
    spans.sort()
    for (s1, e1, _, i1), (s2, _, _, i2) in zip(spans, spans[1:]):
        if s2 < e1: raise ValueError(f'edits {i1} and {i2} overlap; merge them into one edit')
    out, at = [], 0
    for s, e, new, _ in spans:
        out.append(text[at:s]); out.append(new); at = e
    out.append(text[at:])
    return ''.join(out)


def _diff(before, after, path='file'):
    "A unified diff, which is what a person approving an edit should be looking at."
    import difflib
    d = difflib.unified_diff(before.splitlines(), after.splitlines(),
                             f'a/{path}', f'b/{path}', lineterm='', n=2)
    return '\n'.join(d)


def file_tools(host, mx=MAX_TOOL_CHARS):
    "Reading and editing files, by exact text or by hash-verified address."

    def add_root(path: str) -> str:
        """Open another folder, so it can be read and written like the ones already open.

        Ask for this only when the user has named a folder outside the open ones. It widens what
        you may change on their machine, so it goes to them for approval like any other write.
        """
        try: return f'opened {host.add_root(path)}. Open folders: ' + ', '.join(host.roots)
        except Exception as e: return err(f'could not open {path}', e)

    def view_file(path: str, start: int = 0, end: int = 0) -> str:
        """Read a file as `lineno|hash|content` lines. Optionally limit to lines `start`..`end`.

        Always read this way before editing: `edit_file` addresses lines by the exact
        hashes this returns. The view is also the address book.
        """
        from exhash import lnhashview, lnhashview_file
        p = readable(host, path)
        # through the host, so a host answering from somewhere other than disk -- an editor
        # holding an unsaved buffer -- is seen as itself rather than as a stale copy. The
        # hashes are the same either way: both views hash `line_hash(lineno, line)` over the
        # same lines. `replace_text` and `create_file` go through the host too; `edit_file`
        # verifies against the file, so against such a host it refuses rather than misfires
        # a host is allowed to raise from any method, and one may not implement `read` at all;
        # that is a reason to read the file, not to lose the tool
        try: text = host.read(str(p))
        except Exception: text = None
        if text is None and not p.exists(): return err(f'no such file: {p}')
        view = str(lnhashview(text, start or None, end or None) if text is not None
                   else lnhashview_file(str(p), start or None, end or None))
        return clip_lines(view.splitlines(), start=(start or 1), n=mx,
                          more='call view_file(path, start={next}) to continue')

    def replace_text(path: str, edits: str) -> str:
        """Edit a file by exact text replacement, and return the diff. Usually the easier editor.

        `edits` is a JSON array of objects, applied together:
          [{"oldText": "def old(a):", "newText": "def new(a, b):"},
           {"oldText": "return a", "newText": "return a + b"}]

        Rules, all of them checked *before* anything is written. A rejected edit leaves
        the file exactly as it was:

        - Every `oldText` must appear **exactly once** in the file. If it appears twice,
          include more surrounding lines until it is unique. Do not guess which one.
        - Every `oldText` is matched against the file as it is **now**, not against the
          result of the earlier edits in the same call. Overlapping or nested spans are
          refused. Merge them into one edit instead.
        - Keep `oldText` as short as it can be while still unique. Do not paste a whole
          function to change one line of it.
        - An empty `oldText` is refused. To create a file use `create_file`. To append,
          include the last existing line in `oldText`.

        This and `edit_file` do the same job by different addresses: `edit_file` names
        lines by hash, which catches a stale read but costs a `view_file` before every
        edit and again after each one. Prefer this for ordinary edits. Prefer `edit_file`
        when you must be certain the line you are changing is the line you read.
        """
        p = host.check(path)
        if hasattr(host, 'check_write'): host.check_write(p)
        try: items = _edits(edits)
        except Exception as e: return err('could not parse edits', e)
        if not items: return err('no edits given')
        try: before = host.read(str(p))
        except Exception as e: return err(f'could not read {p}', e)
        if before is None: return err(f'no such file: {p}. Use create_file to create it')
        try: after = _apply_edits(before, items)
        except ValueError as e: return err(str(e))
        if after == before: return err('the edits changed nothing; check oldText against a fresh view_file')
        try: host.write(str(p), after)
        except Exception as e: return err('write failed', e)
        return clip(f'replaced {len(items)} block(s) in {p}\n' + _diff(before, after, str(p)), mx)

    def edit_file(path: str, commands: str) -> str:
        """Edit a file with hash-verified exhash commands, and return the diff.

        `commands` is a JSON array of command arrays, each starting with an address taken
        from `view_file`, e.g.
          [["12|a1b2|", "s", "old text", "new text"],
           ["30|9f3c|", "a", "a new line appended after line 30"]]
        Every address's hash is checked immediately before it runs. An edit built on a
        stale view fails instead of damaging the wrong line. Nothing is written unless
        every command succeeds.
        """
        from exhash import file_exhash
        p = host.check(path)
        if hasattr(host, 'check_write'): host.check_write(p)
        try: cmds = _cmds(commands)
        except Exception as e: return err('could not parse commands', e)
        if not cmds: return err('no commands given')
        try: return clip(str(file_exhash(str(p), *cmds)), mx)
        except Exception as e: return err('edit failed', e)

    def create_file(path: str, text: str = '') -> str:
        "Create (or overwrite) a whole file. For changes to an existing file prefer `replace_text`."
        try: return f'wrote {host.write(path, text)}'
        except Exception as e: return err('write failed', e)

    return [view_file, replace_text, edit_file, create_file, add_root]

A real file on disk, and a host that lets the tools reach it:

In [ ]:
p = tmp/'greet.py'
p.write_text('def greet(name):\n    return "hi " + name\n')
view_file, replace_text, edit_file, create_file, add_root = file_tools(NullHost([str(tmp)]))
print(view_file(str(p)))

1|2337|def greet(name):
2|0431|    return "hi " + name


An edit quotes an address from that view. Nothing is written unless every command succeeds, and the diff is what comes back.

In [ ]:
line2 = view_file(str(p)).splitlines()[1]
addr = '|'.join(line2.split('|')[:2]) + '|'
print(edit_file(str(p), json.dumps([[addr, 's', '"hi "', '"hello "']])))

--- /var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmp1hnmtoaq/greet.py
+++ /var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmp1hnmtoaq/greet.py
 1|2337|def greet(name):
-2|0431|    return "hi " + name
+2|709e|    return "hello " + name



`view_file` reads through `Host.read` rather than off the disk directly, so a host that answers
from somewhere else -- an editor holding an unsaved buffer, a `MemHost` holding no file at all --
is *seen* as it really is. `lnhashview` over that text and `lnhashview_file` over the same bytes
produce the same hashes, so nothing about addressing changes.

Where such a host's text is not also on disk, the two editing tools part company, and the
difference is worth knowing. `replace_text` and `create_file` go through `Host.read`/`Host.write`,
so they edit and return the host's text. `edit_file` verifies its addresses against the file, so
it *refuses* -- a stale-hash error rather than a write to the wrong line. That is the safe half of
the trade, and it only arises for text the host has not written down.

In [ ]:
#: a host whose text is not what is on disk is viewed as itself, and edited safely
class BufferHost(NullHost):
    "A host whose answer for one path is a buffer, the way an editor's would be."
    def __init__(self, roots, buffers): super().__init__(roots); self.buffers = dict(buffers)
    def read(self, path): return self.buffers.get(str(path), super().read(path))
    def write(self, path, text): self.buffers[str(path)] = str(text); return str(path)

stale = tmp/'stale.py'
stale.write_text('ON_DISK = 0\n')
bh = BufferHost([str(tmp)], {str(stale): 'IN_BUFFER = 1\n'})
vf, rt, ef, cf, _ = file_tools(bh)

seen = vf(str(stale))
test_eq('IN_BUFFER' in seen, True)
test_eq('ON_DISK' in seen, False)

In [ ]:
#: the exact-text tools work on that same buffer, and leave the disk alone
test_eq(rt(str(stale), json.dumps([['IN_BUFFER = 1', 'IN_BUFFER = 2']])).startswith(ERR), False)
test_eq(bh.buffers[str(stale)], 'IN_BUFFER = 2\n')
test_eq(stale.read_text(), 'ON_DISK = 0\n')

In [ ]:
#: and the hash-addressed one refuses rather than writing over the line it cannot see
addr = '|'.join(vf(str(stale)).splitlines()[0].split('|')[:2]) + '|'
out = ef(str(stale), json.dumps([[addr, 's', '2', '3']]))
test_eq(out.startswith(ERR), True)
test_eq('stale lnhash' in out, True)
test_eq(bh.buffers[str(stale)], 'IN_BUFFER = 2\n')

In [ ]:
test_eq(p.read_text(), 'def greet(name):\n    return "hello " + name\n')
addr

'2|0431|'

An address whose hash no longer matches is refused. This is the whole point of the scheme: the model's view of line 2 is verified against the file as it is now, immediately before the edit runs.

In [ ]:
edit_file(str(p), json.dumps([['2|0000|', 's', 'hello', 'howdy']]))

'ERROR: edit failed: ValueError: stale lnhash at line 2: expected 0000, got 709e (line changed since your view)'

## Notebooks

The harness deliberately does not own a notebook representation: exhash addresses cells by path and id without one. Only the two operations that genuinely need to know what a notebook *is* are delegated to the host.

In [ ]:
#| export
def notebook_tools(host, mx=MAX_TOOL_CHARS):
    "Notebooks, addressed by cell id rather than by line."

    def notebook_cells(path: str) -> str:
        "List a notebook's cells: id, type, and first line. Cell ids are what `edit_cell` addresses."
        try: rows = host.nb_cells(str(readable(host, path)))
        except NotImplementedError: raise
        except Exception as e: return err('could not read notebook', e)
        return clip('\n'.join(f'{i}  {t:8} {(s or "").strip().splitlines()[0][:100] if (s or "").strip() else ""}'
                              for i, t, s in rows) or '(empty notebook)')

    def view_cell(path: str, cell_id: str) -> str:
        "Read one notebook cell as `lineno|hash|content` lines, ready to address with `edit_cell`."
        from exhash import lnhashview_cell
        try: return clip(str(lnhashview_cell(str(readable(host, path)), cell_id)))
        except Exception as e: return err('could not read cell', e)

    def edit_cell(path: str, cell_id: str, commands: str) -> str:
        "Edit one notebook cell's source with exhash commands from `view_cell`. Same format as `edit_file`."
        from exhash import cell_exhash
        try: cmds = _cmds(commands)
        except Exception as e: return err('could not parse commands', e)
        try: return clip(str(cell_exhash(str(host.check(path)), cell_id, *cmds)))
        except Exception as e: return err('edit failed', e)

    def add_cell(path: str, source: str, index: int = -1, cell_type: str = 'code') -> str:
        "Insert a new cell into a notebook at `index` (-1 appends). Creates the notebook if needed."
        try: return f'added cell {host.nb_add_cell(str(host.check(path)), source, int(index), cell_type)} to {path}'
        except NotImplementedError: raise
        except Exception as e: return err('could not add cell', e)

    return [notebook_cells, view_cell, edit_cell, add_cell]

A host without notebook support raises `NotImplementedError`, and `tools_for` omits the notebook tools.

In [ ]:
notebook_cells, view_cell, edit_cell, add_cell = notebook_tools(host)
with expect_fail(NotImplementedError): notebook_cells('/proj/x.ipynb')
[t.__name__ for t in notebook_tools(host)]

['notebook_cells', 'view_cell', 'edit_cell', 'add_cell']

## The web and remembered research

Two groups, and the split matters: the web tools go out now, while the memory tools recall pages read earlier as whole document sections. A question that was researched last week should cost a memory search rather than another crawl.

### Generating a picture

`gpt-image-1` uses `/v1/images/generations` rather than `Chat`. A model that cannot draw calls this tool. The tool saves each picture beside the session and returns its path.

The tool is registered only when `OPENAI_API_KEY` is set.

In [ ]:
#| export
def media_dir(session=''):
    "Where a turn's generated pictures are kept: beside the session record that produced them."
    d = Path(session or '.') / 'media'
    d.mkdir(parents=True, exist_ok=True)
    return d

def mime_for(path):
    "The mime a saved picture is: what its bytes say, else what its name does."
    try: m = detect_mime(Path(path).read_bytes()[:64])
    except OSError: m = None
    return m or mimetypes.guess_type(str(path))[0] or 'image/png'

def save_media(m, session='', stem='image'):
    "One `{'mime','data'}` from `Resp.media` written under the session, as a `Path`."
    mime = m.get('mime') or 'image/png'
    ext = mimetypes.guess_extension(mime) or '.' + mime.split('/')[-1]
    d = media_dir(session)
    n = 1 + len(list(d.glob(f'{stem}-*')))
    p = d / f'{stem}-{n}{ext}'
    p.write_bytes(m['data'])
    return p

#: Where a picture is asked for. The first is any chat model that draws through the Responses
#: image tool; the second is the dedicated endpoint, for models that cannot draw at all.
RESPONSES_API = 'https://api.openai.com/v1/responses'
IMAGE_API = 'https://api.openai.com/v1/images/generations'
IMAGE_MODEL = 'gpt-image-1'
IMAGE_SIZES = ('1024x1024', '1536x1024', '1024x1536', 'auto')

def image_available(): return bool(os.environ.get('OPENAI_API_KEY'))

def _hdrs(): return {'Authorization': f"Bearer {os.environ['OPENAI_API_KEY']}"}

def _post_image(prompt, size, n, timeout=120, model=IMAGE_MODEL):
    "Raw `data` rows from the images endpoint."
    import httpx
    r = httpx.post(IMAGE_API, timeout=timeout, headers=_hdrs(),
                   json={'model': model, 'prompt': prompt, 'size': size, 'n': n})
    r.raise_for_status()
    return r.json().get('data') or []

#: Vendor prefixes a spec carries and the endpoint does not: `openai/gpt-5.6-luna` is a
#: `model_not_found` at the API, which spells the same model `gpt-5.6-luna`.
API_VENDORS = ('openai/', 'azure/')

def api_model(model):
    "A spec's `model_id` as the OpenAI endpoints spell it."
    s = str(model or '')
    for v in API_VENDORS:
        if s.startswith(v): return s[len(v):]
    return s

def _post_responses(prompt, model, timeout=300):
    "The untouched Responses reply from `model` drawing through its built-in image tool."
    import httpx
    r = httpx.post(RESPONSES_API, timeout=timeout, headers=_hdrs(),
                   json={'model': api_model(model), 'input': prompt,
                         'tools': [{'type': 'image_generation'}]})
    r.raise_for_status()
    return r.json()

def draws_itself(spec):
    """Can `spec`'s own model draw, given the image tool?

    `supported_output_modalities` says no for every chat model: it describes what one returns
    unprompted. `Caps.tools` is the second answer, and it is what makes gpt-5.6-luna draw as
    itself rather than handing the prompt to a different model."""
    if spec is None: return False
    c = spec_caps(spec)
    return bool(c is not None and 'image' in getattr(c, 'tools', ()))

def _from_responses(raw):
    "Generated pictures out of a Responses reply, read by the same `rishi` code a turn uses."
    from rishi.remote import gen_media
    return gen_media(raw)

def image_tools(host, mx=MAX_TOOL_CHARS, session='', get_spec=None, on_media=None):
    "Drawing: by the turn's own model where it can, and by the images endpoint where it cannot."

    def generate_image(prompt: str, size: str = '1024x1024', n: int = 1, model: str = '') -> str:
        """Generate a picture from a description and save it. Returns the paths written.

        Use whenever the user asks for an image. `size` is one of 1024x1024, 1536x1024,
        1024x1536, or auto. Set `model` to use the dedicated images endpoint.
        """
        if not image_available(): return err('image generation is unavailable', 'OPENAI_API_KEY is not set')
        if size not in IMAGE_SIZES: return err('unknown size', f'{size!r}; use one of {", ".join(IMAGE_SIZES)}')
        spec = get_spec() if get_spec else None
        try:
            if not model and draws_itself(spec): media = _from_responses(_post_responses(prompt, spec.model_id))
            else: media = [{'mime': 'image/png', 'data': b64decode(r['b64_json'])}
                           for r in _post_image(prompt, size, max(1, min(int(n or 1), 4)),
                                                model=api_model(model or IMAGE_MODEL))
                           if r.get('b64_json')]
        except Exception as e: return err('could not generate the image', e)
        if not media: return err('the model returned no image', 'the reply carried no picture')
        try: out = [save_media(m, session, 'generated') for m in media]
        except Exception as e: return err('could not save the image', e)
        if on_media: on_media(out)
        return clip('\n'.join(str(p) for p in out))

    return [generate_image]

In [ ]:
import os as _os
_k = _os.environ.pop('OPENAI_API_KEY', None)
try:
    test_eq(image_available(), False)
    gi = image_tools(None)[0]
    assert failed(gi('a cat')) and 'OPENAI_API_KEY' in gi('a cat')   # refuses, never calls
finally:
    if _k is not None: _os.environ['OPENAI_API_KEY'] = _k

assert failed(image_tools(None)[0]('a cat', size='4096x4096'))       # size checked before the call

import tempfile
with tempfile.TemporaryDirectory() as d:
    p = save_media({'mime': 'image/png', 'data': b'\x89PNG\r\n\x1a\nx'}, d, 'generated')
    test_eq((p.name, p.parent.name), ('generated-1.png', 'media'))

In [ ]:
_key = os.environ.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = 'test'
_seen = {}
def _fake_image(prompt, size, n, timeout=120, model=IMAGE_MODEL):
    _seen['model'] = model
    return [{'b64_json': 'iVBORw0KGgp4'}]
_orig_post_image = _post_image
_post_image = _fake_image
try:
    with tempfile.TemporaryDirectory() as d:
        out = image_tools(None, session=d)[0]('a cat', model='openai/gpt-image-2')
        assert not failed(out)
    test_eq(_seen['model'], 'gpt-image-2')
finally:
    _post_image = _orig_post_image
    if _key is None: os.environ.pop('OPENAI_API_KEY', None)
    else: os.environ['OPENAI_API_KEY'] = _key

In [ ]:
#| export
def web_tools(host, mx=MAX_TOOL_CHARS):
    "The web, for the questions whose answer depends on current documentation."

    def web_search(query: str) -> str:
        "Search the web. Returns titles and urls. Follow up with `read_url` on the useful ones."
        docs = host.web_search(query, n=MAX_HITS)
        if not docs: return f'no results ({host.research_note})'
        return clip('\n'.join(f'{d.title}\n  {d.url}' for d in docs))

    def read_url(url: str, remember: bool = True) -> str:
        """Read one web page as markdown. A GitHub file, an arxiv paper or a YouTube
        transcript is read as what it is rather than as the page around it.

        It enters durable research memory by default. Pass `remember=False` for sensitive,
        obviously irrelevant, or exploratory results that should remain ephemeral.
        """
        d = host.read_url(url, remember=remember)
        return clip(d.text if d else f'could not read {url} ({host.research_note})')

    def research(query: str) -> str:
        "Search the web and read the top results into one cited digest. Slower than `web_search`. Use for depth."
        return clip(host.research(query) or f'nothing found ({host.research_note})')

    return [web_search, read_url, research]

In [ ]:
#| export
def memory_tools(host, mx=MAX_TOOL_CHARS):
    "Durable pages and research recalled as document sections rather than flat snippets."

    def memory_search(query: str, limit: int = 8) -> str:
        """Search pages remembered from earlier reads and research.

        Returns whole operative sections with breadcrumbs plus related semantic paths. Use
        this before searching the live web when the question may have been researched before.
        """
        try: return clip(json.dumps(host.memory_search(query, int(limit)), default=str), MAX_TOOL_CHARS * 2)
        except Exception as e: return err('memory search failed', e)

    def memory_tree(document: str = '') -> str:
        """Browse remembered document headings without embedding a query.

        `document` may be a title substring or stable document id. Leave it empty to list
        all remembered roots, then call again with the relevant document.
        """
        try: return clip(json.dumps(host.memory_tree(document), default=str), MAX_TOOL_CHARS * 2)
        except Exception as e: return err('memory tree failed', e)

    def memory_read(node_id: str) -> str:
        "Read one whole remembered section by the node id returned by memory_search/tree."
        try: return clip(json.dumps(host.memory_read(node_id), default=str), MAX_TOOL_CHARS * 3)
        except Exception as e: return err('memory read failed', e)

    def memory_topics(limit: int = 12) -> str:
        "Map remembered material into labelled semantic clusters and representative members."
        try: return clip(json.dumps(host.memory_topics(int(limit)), default=str), MAX_TOOL_CHARS * 2)
        except Exception as e: return err('memory topics failed', e)

    def ask_memory(question: str, document: str = '', instruction: str = '') -> str:
        """Ask remembered research a question and get a short cited answer, not the sections.

        Prefer this to `memory_search` when you want an answer rather than material: the search
        returns whole sections into your context and this returns a paragraph, which is the same
        trade `delegate_search` makes. `document` narrows it to one remembered document by title
        or id.

        Some of what the vault holds is private. A statement, a medical letter, an exported
        chat. Those are answered by a model on this machine that is instructed not to repeat any
        personal detail to you. What you get back is shape and quantity: how many, what kind,
        which period, whether two things agree. It will tell you what it is holding and what
        instruction would let it answer usefully. Send that back as `instruction` and it gets
        another turn on the same material.

        Do not ask it for the details it withheld, or ask it to relay them "for the user". It
        will refuse, and the refusal is the point rather than an obstacle.
        """
        try: r = host.ask(question, ref=document or None, instruction=instruction)
        except NotImplementedError: raise
        except Exception as e: return err('could not ask memory', e)
        rows = [str(r.get('answer') or '(no answer)')]
        if (p := r.get('pii')) and p.get('has_pii'):
            rows.append(f"\n[answered on a local model; it holds back "
                        f"{', '.join(sorted(p.get('identifying') or {}))}. Reply with `instruction=` "
                        f"to say what you need -- a count, a total, a comparison, a yes or no.]")
        if (c := r.get('cited')):
            rows.append('\n' + '\n'.join(f"[{x['n']}] {x['breadcrumb']}  ({x['node_id']})" for x in c))
        return clip('\n'.join(rows), MAX_TOOL_CHARS * 2)

    def memory_forget(doc_id: str) -> str:
        """Purge one bad, sensitive, stale or irrelevant remembered document by id.

        This removes its tree, chunks and ANN entries. Use only when the user requests it. Do not silently curate their memory.
        """
        try: return 'forgot document' if host.memory_forget(doc_id) else 'document was not forgotten'
        except Exception as e: return err('memory purge failed', e)

    tools = [memory_search, memory_tree, memory_read, memory_topics, memory_forget]
    # `ask` is a model call rather than a lookup. It is not part of the group
    if _supports(host, 'ask'): tools.insert(4, ask_memory)
    return tools

In [ ]:
#| export
def api_tools(host, mx=MAX_TOOL_CHARS):
    "Read an API specification, browse what it declares, and call one operation."

    def api_load(src: str, name: str = '') -> str:
        """Load an OpenAPI or discovery document from a url or a path.

        Do this before `api_ops` or `api_call`. `src` is often `<host>/openapi.json`. Returns
        the operation count and the groups, which is what to narrow by next.
        """
        try: return clip(json.dumps(host.api_load(src, name), default=str), mx)
        except Exception as e: return err('could not load the spec', e)

    def api_ops(group: str = '', name: str = '', match: str = '', offset: int = 0) -> str:
        """List the operations a loaded spec declares, with their signatures.

        Narrow with `group` or `match` first: a real API has hundreds of operations, and
        reading all of them is not how you find the one you want. One page comes back at a
        time. `api_load` says how many there are in total, and `offset` walks the rest.
        """
        try:
            rows = host.api_ops(group, name, match, offset=offset)
            total = host.api_count(group=group, name=name, match=match)
            out = {'operations': rows}
            if offset or len(rows) < total:
                out |= {'matched': total, 'showing': f'{offset + 1}-{offset + len(rows)}',
                        'more': f'call again with offset={offset + len(rows)}'} if offset + len(rows) < total else {'matched': total}
            return clip(json.dumps(out, default=str), mx)
        except Exception as e: return err('could not read the operations', e)

    def api_call(operation: str, name: str = '', params: dict = None) -> str:
        """Call one operation, passing `params` under the names `api_ops` reported.

        A parameter the operation does not declare is an error rather than an extra query
        field, which is what makes a wrong call fail loudly instead of quietly.
        """
        try: return clip(json.dumps(host.api_call(operation, name, **(params or {})), default=str), mx)
        except Exception as e: return err(f'{operation} failed', e)

    return [api_load, api_ops, api_call]

In [ ]:
[t.__name__ for t in web_tools(host)], [t.__name__ for t in memory_tools(host)]

(['web_search', 'read_url', 'research'],
 ['memory_search',
  'memory_tree',
  'memory_read',
  'memory_topics',
  'memory_forget'])

`NullHost.web_search` returns an empty result instead of raising. The tool remains available, and `research_note` identifies the backend that answered.

In [ ]:
web_search, read_url, research = web_tools(host)
web_search('nbdev v3 export'), memory_tools(host)[0]('nbdev')

('no results ()', 'ERROR: memory search failed: NotImplementedError: ')

## The live session

The session tools expose the user's kernel and terminal. `run_python` writes to the live namespace and requires approval. `inspect_python` cannot change the user's variables and needs no approval. `read_terminal` shows existing terminal output but cannot run commands.

`ask_memory` makes a model call inside a tool call. It returns an answer without loading every candidate section into the caller's context. A host with private documents answers on a separate local model and returns only what that model permits. The tool appears only when the host implements `Host.ask`.

`memory_tools` recalls prior reads. `watch_tools` schedules future reads. Both use the same store, so a fired reminder becomes an ordinary searchable note.

In [ ]:
#| export
def watch_tools(host, mx=MAX_TOOL_CHARS):
    "Standing interests: what to put back on the desk later, and what has come due now."

    def remember(text: str, title: str = '', tags: str = '') -> str:
        """Write a conclusion into durable memory so a later session finds it.

        For what you worked out, not for what you read. `read_url` already files pages.
        `tags` is a comma-separated list.
        """
        try:
            d = host.remember(text, title=title or None,
                              tags=[t.strip() for t in tags.split(',') if t.strip()])
            return f"remembered {d.get('title')!r} as {d.get('doc_id')}"
        except Exception as e: return err('could not remember', e)

    def set_reminder(text: str, every: str = '1w', note: str = '') -> str:
        """Come back to `text` every `every` ('30m', '6h', '1d', '1w').

        The reminder files itself into memory when it comes due. It surfaces in
        `memory_search` and in `poll_watches` rather than needing a notification channel.
        """
        try:
            w = host.watch(text, action='remind', every=every, note=note or None)
            return f"reminder {w['id']} set, every {every}"
        except Exception as e: return err('could not set reminder', e)

    def watch_url(url: str, every: str = '1d', note: str = '') -> str:
        "Re-read `url` every `every` and file each version in memory. Changes are visible over time."
        try:
            w = host.watch(url, action='url', every=every, note=note or None)
            return f"watching {url} as {w['id']}, every {every}"
        except Exception as e: return err('could not watch', e)

    def list_watches(due_only: bool = False) -> str:
        "Every standing watch and reminder, soonest first. `due_only` shows just what has come due."
        try:
            ws = host.watches(due_only=bool(due_only))
            if not ws: return 'nothing is being watched'
            return clip('\n'.join(
                f"{w['id']}  {w['action']:8} every {int(w['every'])}s  runs={w['runs']}"
                f"  {w.get('last_status') or 'never run'}  {str(w['target'])[:80]}" for w in ws))
        except Exception as e: return err('could not list watches', e)

    def cancel_watch(watch_id: str) -> str:
        "Delete one watch by id. Only when the user asks. Do not silently curate their reminders."
        try:
            host.unwatch(watch_id)
            return f'cancelled {watch_id}'
        except Exception as e: return err('could not cancel', e)

    def poll_watches() -> str:
        """Run every watch that has come due, and report what fired.

        Call this when the user asks what is outstanding, or at the start of a session.
        Anything that fired is now in memory: follow up with `memory_search`.
        """
        try:
            r = host.poll()
            if not r.get('ran'): return f"nothing due ({r.get('checked', 0)} watched)"
            lines = [f"{x['status']:7} {x['action']:8} {str(x['target'])[:90]}" for x in r['results']]
            return clip(f"{r['ran']} of {r['checked']} fired\n" + '\n'.join(lines))
        except Exception as e: return err('poll failed', e)

    return [remember, set_reminder, watch_url, list_watches, cancel_watch, poll_watches]

In [ ]:
#| export
def session_tools(host, mx=MAX_TOOL_CHARS):
    "The live kernel the user is working in, and the terminal they are looking at."

    def list_vars() -> str:
        "List the variables visible in the user's live session: name, type, and a short value."
        return clip(host.list_vars() or '(empty session)')

    def run_python(code: str) -> str:
        """Run Python in the user's live kernel namespace.

        Read any variable freely. Bind results to NEW names so they survive to the next
        call. Mutating or deleting the user's variables is refused. Rebind instead
        (`df2 = df.drop(...)`). Call `list_vars` first if you do not know what is there.
        """
        try: return clip(host.run_python(code))
        except NotImplementedError: raise
        except Exception as e: return err('run failed', e)

    def inspect_python(code: str, scope: str = 'isolated') -> str:
        """Look at the user's live variables by running Python that cannot change them.

        Two scopes. Both leave the user's variables exactly as they were. They differ in
        how much Python you get. Pick by what the question needs:

        - `scope='isolated'` (default) runs in an allowlist sandbox on a copy. Attribute
          reads and builtins work. `df.shape`, `len(df)`, `type(x).__name__`. And most
          library method calls are refused. Costs nothing to be wrong about.
        - `scope='overlay'` runs the real interpreter against the real namespace. Library
          calls work: `list(df.columns)`, `df.head(3).to_dict()`, `model.summary()`. Names
          you bind persist into your own layer for later calls. You still cannot delete,
          rebind or mutate anything the user made. That is refused, with an explanation.

        Start isolated. Move to overlay when the sandbox refuses something you need. Neither
        needs approval, and both run while one of the user's cells is still going. For work
        that must land in the *user's* namespace, use `run_python` instead.
        """
        try: return clip(host.inspect_python(code, scope=scope))
        except NotImplementedError: raise
        except Exception as e: return err('inspection failed', e)

    def read_terminal(lines: int = 200) -> str:
        """Read what the IDE's terminal has printed. A failing build, a stack trace, a test run.

        This is *read only*: it shows what the user ran, and cannot run anything. Use it
        when they mention an error they are looking at rather than asking them to paste it.
        """
        return clip(host.terminal_text(int(lines)) or 'the terminal has printed nothing yet')

    # No tool per recipe: `run_python` composes one in a line. See `coding_patterns`.
    return [list_vars, run_python, inspect_python, read_terminal]

In [ ]:
#| export
def shell_tools(host, mx=MAX_TOOL_CHARS):
    "Running a command, which is the only way to find out whether the work is done."

    def run_shell(command: str, cwd: str = '', timeout: int = 120) -> str:
        """Run one shell command in the project and return its exit code and output.

        This is how you check your work, and you are expected to use it: after an edit, run
        the tests. After a change to a signature, run the type checker or the linter the
        project already uses. Before saying something passes, make it pass here. A claim
        with no command behind it is a guess, and will be read as one.

        - stdout and stderr come back interleaved, as a person would see them, with the
          exit code on the first line. A non-zero exit is a *result*: read the output and
          fix the cause, do not run it again unchanged.
        - `cwd` defaults to the first open folder and must stay inside the open folders.
        - `timeout` is in seconds. The command is killed when it expires. Do not start
          servers, watchers, REPLs, or anything else that does not exit on its own.
        - Use the project's own commands. The ones in its README, `pyproject.toml`, or
          `Makefile`. Rather than a global tool that may not be what it uses.
        - This may be put to the user for approval. Send one purposeful command rather
          than a chain of exploratory ones.
        """
        cmd = str(command or '').strip()
        if not cmd: return err('no command given')
        try: code, out = host.run_cmd(cmd, cwd=(str(cwd).strip() or None), timeout=int(timeout))
        except NotImplementedError: raise
        except Exception as e: return err('command could not be run', e)
        head = f'exit {code}' + ('' if code == 0 else '  (command FAILED)')
        body = clip((out or '').rstrip() or '(no output)', mx - 200,
                    more='re-run narrowing the command (a single test, `| tail -50`) rather than repeating it')
        return f'{head}\n{body}' if code == 0 else f'{ERR}{head}\n{body}'

    return [run_shell]

In [ ]:
list_vars, run_python, inspect_python, read_terminal = session_tools(host)
run_python('df2 = df.dropna()'), host.ran

('ok', ['df2 = df.dropna()'])

There is no `scale_numeric` here any more. It was a pandas min-max scaler written out as a tool. One library's one transformation, permanently in the tool list of a harness that has no idea what language the project is written in. `run_python` composes it in a line, and `coding_patterns` is explicit that a general harness should not carry a tool per recipe.

In [ ]:
run_python("df_norm = (df - df.min()) / (df.max() - df.min()).replace(0, 1)"), host.ran[-1]

('output must be a variable name', 'df2 = df.dropna()')

## Skills as tools

`read_skill` is what makes the index in the system prompt affordable: names and descriptions go out with every turn, bodies only when asked for. `create_skill` writes a project-local `SKILL.md`, and never overwrites one.

In [ ]:
#| export
def skill_tools(host, get_skills, mx=MAX_TOOL_CHARS):
    "Reading discovered skills and creating project-local Agent Skills."

    def read_skill(name: str) -> str:
        """Read one skill in full: how to use a tool or a library that is already installed here.

        The skill list in your briefing gives names and one-line descriptions. Read the
        matching one *before* doing the work it describes, not after it has gone wrong.
        """
        ss = get_skills()
        s = find(ss, name)
        if s is None:
            return f'no skill matching {name!r}. Available: ' + ', '.join(x.name for x in ss)
        return clip(f'<skill name="{s.name}" from="{s.where}">\n{s.text()}\n</skill>', MAX_TOOL_CHARS * 3)

    def create_skill(name: str, description: str, instructions: str) -> str:
        """Create a reusable project skill at `.agents/skills/NAME/SKILL.md`.

        Use this only when the user asks to preserve repeatable project know-how as a
        skill, not for ordinary task notes. `name` must be lowercase kebab-case. `description` says when it applies. `instructions` is the complete Markdown body.
        Existing skills are never overwritten. Run `/reload` after creation to make the
        current agent advertise it immediately.
        """
        from pathlib import Path
        name = str(name or '').strip()
        if not re.fullmatch(r'[a-z0-9]+(?:-[a-z0-9]+)*', name):
            return 'skill name must be lowercase kebab-case (for example, notebook-tests)'
        if not str(description or '').strip(): return 'skill description is required'
        if not str(instructions or '').strip(): return 'skill instructions are required'
        roots = list(host.roots or ())
        if not roots: return 'open a project folder before creating a skill'
        target = Path(host.check(Path(roots[0])/'.agents'/'skills'/name/'SKILL.md'))
        exists = target.exists()
        if not exists:
            try: exists = host.read(str(target)) is not None
            except Exception: pass
        if exists: return f'refusing to overwrite existing skill: {target}'
        title = json.dumps(name, ensure_ascii=False)
        desc = json.dumps(' '.join(str(description).split()), ensure_ascii=False)
        text = f'---\nname: {title}\ndescription: {desc}\n---\n\n{str(instructions).strip()}\n'
        try: host.write(str(target), text)
        except Exception as e: return err('could not create skill', e)
        return f'created {target}; run /reload to load it into the current agent'

    return [read_skill, create_skill]

In [ ]:
read_skill, create_skill = skill_tools(host, lambda: skills)
print(read_skill('notebook-tests')[:120])

<skill name="notebook-tests" from="/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmp1hnmtoaq/.agents/skills/notebook-


An unresolvable name answers with the list, rather than with nothing.

In [ ]:
read_skill('nope')[:80]

"no skill matching 'nope'. Available: coding_patterns, design-taste-frontend, edi"

`create_skill` writes under the first open folder in the layout that `discover` reads. It rejects names that are not lowercase kebab-case before writing.

In [ ]:
create_skill('Notebook Tests', 'when testing', 'body'), create_skill('nb-tests', 'when testing notebooks', 'Use `test_eq`.')

('skill name must be lowercase kebab-case (for example, notebook-tests)',
 'created /proj/.agents/skills/nb-tests/SKILL.md; run /reload to load it into the current agent')

In [ ]:
test_eq(create_skill('nb-tests', 'again', 'body').startswith('refusing to overwrite'), True)
sorted(host.files)

['/proj/.agents/skills/nb-tests/SKILL.md', '/proj/a.py', '/proj/b.py']

## Assembling the tool list

`tools_for` probes each group with a harmless call and drops the whole group when the host does not implement it. Whole groups rather than individual tools, because the groups are the real units of capability: a host with no notebook representation cannot support any of the four notebook tools.

In [ ]:
#| export
def tools_for(host, get_skills=None, extra=(), mx=MAX_TOOL_CHARS, drop=(), get_spec=None, on_media=None):
    """Every tool this host can actually support, plus whatever extensions registered.

    Groups are dropped whole, answered by `Host.capabilities` where the host declares it and by
    a harmless call otherwise. `mx` is what one tool result may spend, a property of the model
    rather than of the tool. `drop` withholds groups the host does support, decided by
    `core.budget_for` and reported by `Agent.budget`.
    """
    drop = set(drop or ())
    tools = []
    if 'code' not in drop: tools += code_tools(host, mx)
    if 'file' not in drop: tools += file_tools(host, mx)
    if 'notebook' not in drop and _has(host, 'notebook', lambda: host.nb_cells('.')): tools += notebook_tools(host, mx)
    if 'web' not in drop and _has(host, 'web', lambda: host.web_search('', n=1)): tools += web_tools(host, mx)
    # credentialled, never probed: a key is either there or it is not
    if 'image' not in drop and image_available(): tools += image_tools(host, mx, get_spec=get_spec, on_media=on_media)
    if 'memory' not in drop and _has(host, 'memory', lambda: host.memory_tree('')): tools += memory_tools(host, mx)
    if 'watch' not in drop and _has(host, 'watch', lambda: host.watches()): tools += watch_tools(host, mx)
    # declared, never probed: `api_ops` raises for a missing spec, not a missing capability
    if 'api' not in drop and _declared(host, 'api'): tools += api_tools(host, mx)
    if 'session' not in drop and _has(host, 'session', lambda: host.list_vars(), lambda: host.terminal_text(1)): tools += session_tools(host, mx)
    shell = _declared(host, 'shell')   # `run_cmd` has no harmless probe. `_supports` asks instead
    if 'shell' not in drop and (_supports(host, 'run_cmd', lambda: host.run_cmd('')) if shell is None else shell):
        tools += shell_tools(host, mx)
    if get_skills is not None and 'skill' not in drop: tools += skill_tools(host, get_skills, mx)
    if 'git' not in drop and getattr(host, 'roots', None) and _declared(host, 'git') is not False:
        from ramabana.git import git_tools
        tools += git_tools(host, mx)
    tools += list(extra or ())
    return tools

A `MemHost` supports code, files and (emptily) the web, but has no notebooks, no memory and no variable listing. It receives only the groups it can actually run:

In [ ]:
ts = tools_for(host)
[t.__name__ for t in ts]

['search_code',
 'grep',
 'ls',
 'similar_code',
 'outline',
 'list_files',
 'view_file',
 'replace_text',
 'edit_file',
 'create_file',
 'web_search',
 'read_url',
 'research',
 'run_shell']

In [ ]:
test_eq(_probe(host, lambda: host.nb_cells('.')), False)
test_eq(_probe(host, lambda: host.search('x')), True)
len(ts), len(tools_for(host, get_skills=lambda: skills))

(14, 16)

A `LocalHost` over a real folder also supports notebooks and a live session. Neither host receives remembered-research tools because neither has a memory index.

In [ ]:
[t.__name__ for t in tools_for(local)]

['search_code',
 'grep',
 'ls',
 'similar_code',
 'outline',
 'list_files',
 'view_file',
 'replace_text',
 'edit_file',
 'create_file',
 'notebook_cells',
 'view_cell',
 'edit_cell',
 'add_cell',
 'web_search',
 'read_url',
 'research',
 'list_vars',
 'run_python',
 'scale_numeric',
 'inspect_python',
 'read_terminal',
 'run_shell']

## Sub-agents

Delegation is a context strategy, not a speed one. A broad question that takes twenty tool calls to answer costs the caller one question and one answer, because the sub-agent's working is discarded with its conversation. A sub-agent gets read-only tools, and cannot delegate further: recursion here is a fan-out tree whose width nobody chose.

In [ ]:
#| export
SUB_MAX_STEPS = 12

SUB_SP = """You are a research sub-agent inside a Python IDE. Another agent has delegated one \
question to you because answering it takes many tool calls and the answer is short.

- Answer exactly the question asked. Nothing else.
- Use your tools as much as you need; nobody is paying attention to how many calls it takes.
- Report what you found, with file paths and line numbers, not what you infer or expect.
- If the answer is that there is nothing, say so plainly. A confident wrong answer is far \
worse than "no matches, and here is what I searched for".
- You cannot edit anything. If the answer implies a change, describe the change and stop.
- `inspect_python` answers questions about the user's live variables without changing them. \
Its default scope is a sandbox that refuses most library calls; pass `scope='overlay'` to \
get the real interpreter. Use it rather than guessing at what is in memory."""


#: Swapped in for the two read-only lines above when a session grants sub-agents writes.
SUB_WRITE_SP = """- You have the delegating agent's write tools as well as its read tools: create and \
edit files, run commands, run Python. Every call is recorded on the session and goes through the \
approval policy the main agent answers to. A refusal comes back with a reason. Read it and change \
the approach.
- Write only what the task asked for. You cannot see the conversation that sent you. Anything \
else you change is a change nobody reviewed.
- Verify with the tool that proves it. Run the test. Read the file back. Report the evidence.
- `run_python` shares the user's kernel namespace. Bind results to NEW names. You cannot rebind or \
delete what the user made."""


def sub_briefing(writes=False):
    "The sub-agent standing instructions, with the read-only sentences swapped out when writes are on."
    if not writes: return SUB_SP
    keep = [ln for ln in SUB_SP.splitlines()
            if not ln.startswith('- You cannot edit anything') and not ln.startswith('- `inspect_python`')]
    return '\n'.join(keep).rstrip() + '\n' + SUB_WRITE_SP


# A sub-agent does not spawn sub-agents: recursion here is a fan-out tree whose width nobody chose.
# Nor does it open standing work: a folder watch outlives the task that opened it, and nobody
# asked for the reviews it would keep producing after the delegation is forgotten.
NO_SUB = frozenset({'delegate_search', 'delegate_parallel',
                    'watch_folder', 'cancel_folder_watch', 'check_folders'})

In [ ]:
#| export
def read_only(tools, max_calls=None, writes=False):
    "The tools a sub-agent may have, optionally behind a hard per-task call budget."
    blocked = NO_SUB if writes else (WRITE_TOOLS | NO_SUB)
    allowed = [t for t in tools if getattr(t, '__name__', '') not in blocked]
    if max_calls is None: return allowed
    state, lock = {'n': 0}, threading.Lock()

    def guarded(f):
        @functools.wraps(f)
        def call(*args, **kw):
            with lock:
                state['n'] += 1
                over = state['n'] > max_calls
            if over:
                return ('Sub-agent tool budget exhausted. Stop calling tools and return the '
                        'best evidence-backed answer now.')
            return f(*args, **kw)
        return call
    return [guarded(t) for t in allowed]

Every write tool and both delegation tools are filtered out, whatever else the host offered:

In [ ]:
[t.__name__ for t in read_only(ts)]

['search_code',
 'grep',
 'ls',
 'similar_code',
 'outline',
 'list_files',
 'view_file',
 'web_search',
 'read_url',
 'research']

In [ ]:
test_eq(set(t.__name__ for t in read_only(ts)) & WRITE_TOOLS, set())
sorted(NO_SUB)

['delegate_parallel', 'delegate_search']

With a budget, the tools themselves stop the loop. Local engines own their internal tool loop. The wrapper is the one hard stop that works on every backend.

In [ ]:
budgeted = read_only(ts, max_calls=1)
search = next(t for t in budgeted if t.__name__ == 'search_code')
search('return'), search('return')

('[memory]\n/proj/a.py:1    def a(): return 1\n  FILE -- use this exact path with view_file/edit_file\n/proj/b.py:1    def b(): return a() + 1\n  FILE -- use this exact path with view_file/edit_file',
 'Sub-agent tool budget exhausted. Stop calling tools and return the best evidence-backed answer now.')

`delegate` runs one question in a throwaway conversation on the same engine, and closes it in a `finally`. A sub-agent whose context leaks back into the session is just a slower way of doing the work inline.

In [ ]:
#| export
def sub_sp(sp=SUB_SP, skills=()):
    "A sub-agent's briefing: its standing instructions, then the bodies of the skills its task named."
    if not skills: return sp
    return sp + '\n\n' + '\n\n'.join(f'## {s.name}\n\n{s.text()}' for s in skills)


def _stopped(run):
    "A stopped delegation answers in text: a dict would reach the model as its own repr."
    return f'The delegated question was stopped ({run.state}) before it answered.'


def delegate(backend, question, tools=(), sp=None, max_steps=SUB_MAX_STEPS, skills=(),
             writes=False,      # hand over WRITE_TOOLS as well
             approve=None,      # the gate those writes answer to, which `spawn` inherits none of
             run=None):         # a pre-registered child run
    "Ask `question` in a throwaway conversation on `backend`'s engine. Returns the answer text."
    sub = None
    run = run or Run(f'run_{uuid.uuid4().hex[:12]}', 'child', str(question), backend.spec.name, current_run())
    if not run.start(): return _stopped(run)
    try:
        # the tool wrappers are the hard stop: native engines own their own tool loop
        kw = {'approve': approve} if approve is not None else {}
        sub = backend.spawn(sp=sub_sp(ifnone(sp, sub_briefing(writes)), skills),
                            tools=read_only(tools, max_calls=max_steps * 4, writes=writes), **kw)
        if hasattr(sub, 'max_steps'): sub.max_steps = max_steps
        if not run.attach(sub): return _stopped(run)
        with run_context(run): out = _delegate_result(sub.send(question, run=run))
        if run.cancelled: return _stopped(run.finish())
        run.finish()
        return out
    except Exception as e:
        run.finish('failed')
        return err('delegation failed', e)
    finally:
        if sub is not None:
            try: sub.close()
            except Exception: pass

In [ ]:
#| export
def delegate_many(backend, questions, tools=(), sp=None, max_steps=SUB_MAX_STEPS, n_workers=4,
                  skills=(), writes=False, approve=None, parent=None):
    "Ask several questions. Register every child before starting serial or parallel workers."
    qs = L(questions)
    if not qs: return L()
    parent = parent or current_run()
    runs = [Run(f'run_{uuid.uuid4().hex[:12]}', 'child', str(q), backend.spec.name, parent,
                getattr(parent, 'grace', .25)) for q in qs]
    def run(item):
        q, child = item
        if child.cancelled:return _stopped(child)
        return delegate(backend, q, tools, sp, max_steps, skills, writes, approve, child)
    items = list(zip(qs, runs))
    # writing sub-agents stay serial. `Approvals` holds one pending ask, and nobody can review
    # concurrent edits to one workspace
    workers = 1 if writes or getattr(backend.spec, 'local', False) else min(n_workers, len(qs))
    ex = concurrent.futures.ThreadPoolExecutor(max_workers=max(1, workers))
    futures = [ex.submit(run, item) for item in items]
    try:
        while True:
            if all(f.done() for f in futures):break
            if parent is not None and parent.cancelled:
                concurrent.futures.wait(futures, timeout=getattr(parent, 'grace', .25))
                break
            time.sleep(.005)
        out = []
        for child, future in zip(runs, futures):
            if future.done():
                try:out.append(future.result())
                except Exception as e:out.append(err('delegation failed', e))
            else:
                child.detach(); out.append(_stopped(child))
        return L(out)
    finally:ex.shutdown(wait=False, cancel_futures=True)

In [ ]:
be = FakeBackend(replies=['the caller never sees this'])
delegate(be, 'which files import fastllm?', tools=ts)

'sub answer'

The spawned conversation is separate, and gone by the time the answer is returned.

In [ ]:
test_eq(len(be.spawned), 1)
be.spawned[0].hist

[{'role': 'user', 'content': 'which files import fastllm?'},
 {'role': 'assistant', 'content': 'sub answer'}]

`delegate_many` keeps the answers in the order the questions were asked. On a local model it runs them one after another on purpose: litert holds one conversation at a time. Fanning out would mean racing for the same engine to find out what happens.

In [ ]:
delegate_many(be, ['what imports fastllm?', 'where is compaction triggered?'], tools=ts)

['sub answer', 'sub answer']

The tools themselves take callables rather than a backend. A model switch mid-session is picked up. The tool the model is holding must not be pinned to whichever backend happened to be current when the list was built.

In [ ]:
#| export
def named_skills(get_skills, names):
    "The skills a delegated task named, and a note about any name that matched nothing."
    if not names or get_skills is None: return [], ''
    every = list(get_skills() or [])
    got, missing = [], []
    for n in [x for x in str(names).replace(',', ' ').split() if x]:
        s = find(every, n)
        got.append(s) if s is not None else missing.append(n)
    if not missing: return got, ''
    return got, (f"\n\n[no skill named {', '.join(missing)}; this repository has "
                 f"{', '.join(s.name for s in every) or 'none'}]")


def subagent_tools(get_backend, get_tools, get_skills=None, get_cloud_backend=None,
                   get_writes=None,     # the session's sub-agent write toggle, read per call
                   get_approve=None):   # the gate those writes answer to
    """The `delegate` tool, bound to whatever backend routing says sub-agents run on.

    Every argument is a callable. A model switch mid-session is picked up. `get_tools` is the
    sub-agent model's tool list, not the turn model's.
    """

    def _writes(): return bool(get_writes()) if get_writes is not None else False
    def _approve(): return get_approve() if (get_approve is not None and _writes()) else None

    def delegate_search(question: str, skills: str = '') -> str:
        """Hand a broad search question to a sub-agent and get back only its conclusion.

        Use this when answering would take many `search_code` / `view_file` / `read_url` /
        `inspect_python` calls whose results you do not need to keep. "where else do we
        do X", "which files import Y", "what shape is everything in this namespace". Its
        working is discarded. The cost to your context is one question and one answer.

        The sub-agent has your read-only tools. Whether it also has your write tools is the
        session's setting rather than yours. With sub-agent writes on it can edit, run commands
        and run Python under the approval policy you answer to. The task you send may then ask
        for a change. With them off it can only report.

        `skills` names skills from your skill index, comma separated, whose text the sub-agent
        should start with: name the one or two its task actually needs. You hold the index and
        it does not. This is the only way it gets a skill without spending a step reading
        one. Leave it empty when the task needs no particular skill.

        Ask one self-contained question. The sub-agent cannot see this conversation.
        """
        b = get_backend()
        if b is None: return 'no model is available to delegate to'
        sk, note = named_skills(get_skills, skills)
        return clip(delegate(b, question, get_tools(), skills=sk, writes=_writes(),
                             approve=_approve()), MAX_TOOL_CHARS) + note

    def delegate_parallel(questions: str, skills: str = '', cloud_model: str = '') -> str:
        """Hand several independent questions to sub-agents at once, and get back every answer.

        `questions` is a JSON array of strings, e.g.
          ["which files import fastllm?", "where is compaction triggered?", "what is df's shape?"]

        Use it when you have two or more questions that do not depend on each other. They
        run concurrently, each in its own throwaway conversation with your read-only tools. Three questions cost you three short answers rather than the sixty tool results
        it would take to answer them yourself. With sub-agent writes on they run one after
        another instead, because their approvals share one queue.

        `skills` names skills from your skill index, comma separated, given to every one of
        them. Use it when the questions share a subject. When they do not, ask them in separate
        `delegate_search` calls so each gets only what its own task needs.

        `cloud_model` optionally selects one configured remote model for this fan-out. It does not
        change the session's turn or default sub-agent model. Every question must be self-contained:
        a sub-agent cannot see this conversation or the other questions.
        """
        b = get_cloud_backend(cloud_model) if cloud_model and get_cloud_backend is not None else get_backend()
        if b is None: return f"no model is available to delegate to{f' ({cloud_model})' if cloud_model else ''}"
        try:
            qs = json.loads(questions) if isinstance(questions, str) else list(questions)
            if not isinstance(qs, list) or not all(isinstance(q, str) for q in qs):
                raise ValueError('expected a JSON array of strings')
        except Exception as e:
            return err('could not parse questions', e)
        if not qs: return 'no questions given'
        sk, note = named_skills(get_skills, skills)
        answers = delegate_many(b, qs, get_tools(), skills=sk, writes=_writes(), approve=_approve())
        return clip('\n\n'.join(f'### {q}\n{a}' for q, a in zip(qs, answers)), MAX_TOOL_CHARS * 2) + note

    return [delegate_search, delegate_parallel]

In [ ]:
delegate_search, delegate_parallel = subagent_tools(lambda: be, lambda: ts)
[t.__name__ for t in subagent_tools(lambda: be, lambda: ts)]

['delegate_search', 'delegate_parallel']

With no model available it says so, rather than raising into the turn.

In [ ]:
test_eq(subagent_tools(lambda: None, lambda: ts)[0]('anything'), 'no model is available to delegate to')
delegate_parallel('["what imports fastllm?"]')

'### what imports fastllm?\nsub answer'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

In [ ]:
#| export
def _delegate_result(text):
    "Reject degenerate delegated prose before it can be presented as research."
    text = str(text or '').strip()
    words = re.findall(r"[A-Za-z0-9_+.-]+", text.lower())
    if not text: return 'Delegated inspection failed: the sub-agent returned no answer.'
    if len(words) >= 12 and max(words.count(w) for w in set(words)) > max(8, len(words) // 4):
        return 'Delegated inspection failed: repetitive output was discarded as unreliable.'
    return text

Parallel delegation registers every child before work starts. Cancelling the parent stops the running child and prevents queued children from spawning.


In [ ]:
class _Sub:
    max_steps = 0
    def __init__(self, owner): self.owner, self.release = owner, threading.Event()
    def send(self, question, run=None):
        self.owner.started.append(question); self.release.wait(); return 'late'
    def cancel(self): self.owner.cancelled += 1; self.release.set(); return True
    def close(self): pass

class _ParentBackend:
    def __init__(self):
        self.spec = AttrDict(name='fake-child', local=False)
        self.started, self.spawned, self.cancelled = [], 0, 0
    def spawn(self, **kw): self.spawned += 1; return _Sub(self)

backend, parent, box = _ParentBackend(), Run('run_parent', grace=.03), []
parent.start()
t = threading.Thread(target=lambda: box.extend(delegate_many(backend, ['a', 'b', 'c'], n_workers=1, parent=parent)), daemon=True)
t.start()
while not backend.started: time.sleep(.001)
test_eq(len(parent.children), 3)
parent.cancel(); t.join(.2)
test_eq((backend.spawned, backend.cancelled), (1, 1))
test_eq(len(box), 3)
# either shape says cancelled, and which one comes back is a matter of whether the worker finished
# inside the join: a future still in flight is detached and reported as its run, one that returned
# carries `delegate`'s own cancellation message. Asserting only the first made this a coin flip
assert all((isinstance(x, dict) and x['state'] in ('cancelled', 'detached'))
           or (isinstance(x, str) and 'cancelled' in x) for x in box)
